# 6.1 CD4 analysis - Simple plots
- Writing notes
- Plot 1: Among shared clones, what is the average distribution of cells belonging to the same clone in CSF?
- Plot 2: Number of cells in CSF per ml over expansion thresholds
- Plot 3: % of cells discovered in CSF over expansion degree
- Plot 4: Permutation of clonotypes - statistical thresholds
- Plot 5: % of CSF-biased clones
- Plot 6: Clonal sharing per cluster
- Plot 7: Column plots to visualize compartment-wise expansion
- Plot 8: Annotating CSF and PBbiased CD4 T cell clones for DEG
- Plot 9: Vulcano plot - DEGs
- Plot 10: The overlap between CD4 and CD8 CSF signatures

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from skbio.diversity.alpha import gini_index, shannon, simpson
from scipy.stats import ttest_ind

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
from matplotlib_venn import venn2
from scipy.stats import spearmanr
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from brokenaxes import brokenaxes

#Pseudobulk gene expression
import decoupler as dc
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

#Statsmodel
from statsmodels.nonparametric.smoothers_lowess import lowess

#Venn
from matplotlib_venn import venn3

#Linear regression
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr

#chi-square
from scipy.stats import chi2_contingency

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

In [ ]:
from importlib.metadata import version
version('scirpy')

In [ ]:
from importlib.metadata import version
version('scipy')

In [ ]:
from importlib.metadata import version
version('matplotlib')

In [ ]:
from importlib.metadata import version
version('seaborn')

In [ ]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("fontTools").setLevel(logging.WARNING)

# Loading data

In [ ]:
sdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_8_CD4_annotated.h5mu")

# Values for writing

In [ ]:
sdata["gex"].obs[["Compartment"]].value_counts()

# Plot 1: Among shared clones, what is the average distribution of cells belonging to the same clone in CSF?

In [ ]:
## First, define clones in CSF
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["clone_handle"].notna()]
obs_df = obs_df[obs_df["Compartment"]=="CSF"]
obs_df = obs_df[obs_df["clone_handle"].notna()]
csf_list = obs_df["clone_handle"].drop_duplicates().tolist()

## First, define clones in PB
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].notna()]
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["Compartment"]=="PB"]
obs_df = obs_df[obs_df["clone_handle"].notna()]
pb_list = obs_df["clone_handle"].drop_duplicates().tolist()

## Take the intersection of the csf and pb lists
shared_list = list(set(csf_list) & set(pb_list))

## Count cells in each Compartment
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["clone_handle"].notna()]
obs_df = obs_df[obs_df["clone_handle"].isin(shared_list)]
count_df = obs_df[["clone_handle", "Compartment"]].value_counts().reset_index()
count_df = count_df.pivot(index="clone_handle", columns="Compartment", values="count").reset_index()

## Convert to percentage in CSF
count_df["total"] = count_df["CSF"]+count_df["PB"]
count_df["csf_pct"] = 100* count_df["CSF"] / count_df["total"]

## Group into expansion bins
count_df["expan0"] = np.where(count_df["total"] > 0, True, False)
count_df["expan5"] = np.where(count_df["total"] > 5, True, False)
count_df["expan10"] = np.where(count_df["total"] > 10, True, False)
count_df["expan15"] = np.where(count_df["total"] > 15, True, False)
count_df["expan20"] = np.where(count_df["total"] > 20, True, False)

# Long-format dataframe
plot_df = pd.concat([
    count_df.loc[count_df["expan0"],  ["csf_pct"]].assign(Group="n > 0"),
    count_df.loc[count_df["expan5"],  ["csf_pct"]].assign(Group="n > 5"),
    count_df.loc[count_df["expan10"], ["csf_pct"]].assign(Group="n > 10"),
    count_df.loc[count_df["expan15"], ["csf_pct"]].assign(Group="n > 15"),
    count_df.loc[count_df["expan20"], ["csf_pct"]].assign(Group="n > 20"),
])

# Plot mean ± SD
plt.figure(figsize=(4,3))

sb.barplot(
    data=plot_df,
    x="Group",
    y="csf_pct",
    errorbar="sd",   # standard deviation
    capsize=.3,
    color="lightgray"
)

plt.ylabel("Mean clone-wise % of cells\nin CSFof shared clones")
plt.xlabel("")
plt.ylim(0, 100)
sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/16_CD4analysis/XX_Mean_clonewise_pct_in_csf_expansion_thresholds.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
# Add mean labels
means = plot_df.groupby("Group")["csf_pct"].mean()
means

## Extended

In [ ]:
## First, define clones in CSF

obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["clone_handle"].notna()]
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["Compartment"] == "CSF"]

csf_list = obs_df["clone_handle"].drop_duplicates().tolist()


## Define clones in PB

obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["Compartment"] == "PB"]
obs_df = obs_df[obs_df["clone_handle"].notna()]

pb_list = obs_df["clone_handle"].drop_duplicates().tolist()


## Intersection of CSF and PB clones

shared_list = list(set(csf_list) & set(pb_list))


## Count cells in each compartment

obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["clone_handle"].notna()]
obs_df = obs_df[obs_df["clone_handle"].isin(shared_list)]

count_df = (
    obs_df[["clone_handle", "Compartment"]]
    .value_counts()
    .reset_index()
)

count_df = (
    count_df
    .pivot(
        index="clone_handle",
        columns="Compartment",
        values="count"
    )
    .reset_index()
)


## Add catpat for each clone

clone_catpat = (
    obs_df[["clone_handle", "catpat"]]
    .drop_duplicates()
)

count_df = count_df.merge(
    clone_catpat,
    on="clone_handle",
    how="left"
)


## Convert to percentage in CSF

count_df["total"] = count_df["CSF"] + count_df["PB"]

count_df["csf_pct"] = (
    100 * count_df["CSF"] / count_df["total"]
)


## Group into expansion bins

count_df["expan0"] = count_df["total"] > 0
count_df["expan5"] = count_df["total"] > 5
count_df["expan10"] = count_df["total"] > 10
count_df["expan15"] = count_df["total"] > 15
count_df["expan20"] = count_df["total"] > 20


## Long-format dataframe

plot_df = pd.concat([
    count_df.loc[
        count_df["expan0"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 0"),

    count_df.loc[
        count_df["expan5"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 5"),

    count_df.loc[
        count_df["expan10"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 10"),

    count_df.loc[
        count_df["expan15"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 15"),

    count_df.loc[
        count_df["expan20"],
        ["csf_pct", "catpat"]
    ].assign(Group="n > 20"),
])


# # Remove all unused categories
for col in plot_df.select_dtypes(include="category").columns:
    plot_df[col] = plot_df[col].cat.remove_unused_categories()


## Set order

plot_df["Group"] = pd.Categorical(
    plot_df["Group"],
    categories=["n > 0", "n > 5", "n > 10", "n > 15", "n > 20"],
    ordered=True
)


## Plot mean ± SD

palette = {
    "NIC": "#FFFFFF",
    "MS": "#000000"
}

fig, ax = plt.subplots(figsize=(5, 3.5))

# Barplot
sb.barplot(
    data=plot_df,
    x="Group",
    y="csf_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    errorbar="sd",
    capsize=.2,
    palette=palette,
    edgecolor="black",
    linewidth=1,
    ax=ax
)

# # Individual clones
# sb.stripplot(
#     data=plot_df,
#     x="Group",
#     y="csf_pct",
#     hue="catpat",
#     hue_order=["NIC", "MS"],
#     dodge=True,
#     palette=palette,
#     size=6,
#     jitter=True,
#     alpha=0.8,
#     edgecolor="black",
#     linewidth=0.5,
#     ax=ax
# )

# Remove duplicate legends
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:2],
    labels[:2],
    title=None,
    frameon=False
)

ax.set_ylabel("Clone-wise % of cells\nin CSF of shared clones")
ax.set_xlabel("")
ax.set_ylim(0, 120)

sb.despine()
plt.tight_layout()

plt.savefig(
    "../../05_plots/16_CD4analysis/XX_Mean_clonewise_pct_in_csf_expansion_thresholds.pdf",
    format="pdf",
    dpi=400
)

plt.show()

In [ ]:
## For writing
plot_df.groupby(["catpat", "Group"], observed=True).size().reset_index().sort_values("Group")

In [ ]:
## statistics
# Store results
stats = []

for group in plot_df["Group"].cat.categories:
    
    # Extract MS and NIC values
    ms = plot_df.loc[
        (plot_df["Group"] == group) &
        (plot_df["catpat"] == "MS"),
        "csf_pct"
    ].dropna()

    nic = plot_df.loc[
        (plot_df["Group"] == group) &
        (plot_df["catpat"] == "NIC"),
        "csf_pct"
    ].dropna()

    # Mann-Whitney U test
    U, p = mannwhitneyu(
        ms,
        nic,
        alternative="two-sided"
    )

    stats.append({
        "Group": group,
        "n_MS": len(ms),
        "n_NIC": len(nic),
        "MS_median": ms.median(),
        "NIC_median": nic.median(),
        "U": U,
        "p_value": p
    })


# Convert to dataframe
stats_df = pd.DataFrame(stats)


# FDR correction (Benjamini-Hochberg)
stats_df["p_adj"] = multipletests(
    stats_df["p_value"],
    method="fdr_bh"
)[1]


# Significance
stats_df["significant"] = stats_df["p_adj"] < 0.05

stats_df

# Plot 2: Number of cells in CSF per ml over expansion thresholds

#### Cells in CSF per ml over expansion thresholds

In [ ]:
## Get obs_df
obs_df = sdata["gex"].obs
obs_df = obs_df[obs_df["catpat"] != "IC"]
# obs_df = obs_df[obs_df["clone_handle"].notna()]

## Select columns
obs_df = obs_df[["donor", "catpat", "clone_handle", "Compartment", "type_x_clone_count"]].drop_duplicates()

# Remove all unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = obs_df[col].cat.remove_unused_categories()

## Pivot
df_input = obs_df.pivot_table(index = ["donor", "catpat", "clone_handle"], columns="Compartment", values="type_x_clone_count", fill_value=0).reset_index()
df_input["total"] = df_input["CSF"] + df_input["PB"]

## Make 5 dataframes at different expansion thresholds
df0 =  df_input[df_input["total"] > 0].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df0["source"] = "0"

df5 =  df_input[df_input["total"] > 5].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df5["source"] = "5"

df10 =  df_input[df_input["total"] > 10].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df10["source"] = "10"

df15 =  df_input[df_input["total"] > 15].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df15["source"] = "15"

df20 =  df_input[df_input["total"] > 20].groupby(["donor", "catpat"], observed=True)["CSF"].sum().reset_index(name="cell_count")
df20["source"] = "20"

### sum_df
sum_df = pd.concat([df0, df5, df10, df15, df20])
sum_df_backup = sum_df.copy()

# Complete donor × source combinations
all_sources = sum_df["source"].unique()

# Complete donor × source combinations
sum_df = (
    sum_df.set_index(["donor", "source"])
      .reindex(
          pd.MultiIndex.from_product(
              [sum_df["donor"].unique(), all_sources],
              names=["donor", "source"]
          )
      )
      .reset_index()
)

# Missing cell counts = 0
sum_df["cell_count"] = sum_df["cell_count"].fillna(0)

# Remove catpat
sum_df = sum_df.drop(columns = ["catpat"])

#Restore catpat
donor_info = sum_df_backup[["donor", "catpat"]].drop_duplicates("donor")
sum_df = sum_df.merge(donor_info, on="donor", how="left")

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
sum_df["csf_vol"] = sum_df["donor"].map(csf_ml)

## cell_count_norm
sum_df["cell_count_norm"] = sum_df["cell_count"] / sum_df["csf_vol"]

## Shift to avoid 0
sum_df["cell_count_norm"] = sum_df["cell_count_norm"] + 1

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
#             plot
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(6.5,5))

# Boxplot (keep white boxes)
sb.boxplot(
    data=sum_df,
    x="source",
    y="cell_count_norm",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=sum_df,
    x="source",
    y="cell_count_norm",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")
ax.axhline(y = 1, color="black")
ax.set_xlabel("Expansion thresholds (n cells > x)")
ax.set_ylabel("CD4 T-cells in CSF/ml") #% of cells in CSF\nnorm. to clone sizes

sb.despine(ax=ax)
plt.tight_layout()

fig.savefig(
    "../../05_plots/16_CD4analysis/XX_CD4_boxplot_counts_of_CSF_biased_clones.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()

#Save a list of expansion thresholds
threshold_order = ["0", "5", "10", "15", "20"]

# Initialize list to collect results
stats_list = []

for xtick in threshold_order:
    data_xtick = sum_df[sum_df["source"] == xtick]
    
    MS_vals = data_xtick[data_xtick["catpat"] == "MS"]["cell_count"]
    ctrl_vals = data_xtick[data_xtick["catpat"] == "NIC"]["cell_count"]
    
    if len(MS_vals) > 0 and len(ctrl_vals) > 0:
        stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
        
        stats_list.append({
            "expansion_threshold": xtick,
            "statistic": stat,
            "p_value": pval,
            "median_MS": MS_vals.median(),
            "median_NIC": ctrl_vals.median(),
            "n_MS": len(MS_vals),
            "n_NIC": len(ctrl_vals)
        })

# Convert to dataframe
df_stats = pd.DataFrame(stats_list)

# Apply Benjamini-Hochberg correction
reject, pvals_bh, _, _ = multipletests(
    df_stats["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

# Add results to dataframe
df_stats["p_value_BH"] = pvals_bh
df_stats["significant_BH"] = reject

df_stats

# Plot 3: % of cells discovered in CSF over expansion degree

In [ ]:
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))

In [ ]:
## Test out making a dataframe first

# Step 1: Data Preparation and Filtering
obs_df = sdata["gex"].obs.copy()

# Make clone_handle categorical
obs_df["clone_handle"] = obs_df["clone_handle"].astype("category")

# Filter for clone_handle in comp_set and remove donors Pt4 and Pt5
# obs_df = obs_df[obs_df["total_clone_count"] > 11]
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[~obs_df["clone_handle"].isna()]
print("I have this many unique clones: ", obs_df["clone_handle"].nunique(), ", across both CSF and PB combined")

# Filter of CSF to make certain that we are working with CSF clones
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories()
obs_df["clone_handle"] = obs_df["clone_handle"].cat.remove_unused_categories()
obs_df["Compartment"] = obs_df["Compartment"].cat.remove_unused_categories()

#Polish dataframe
obs_df = obs_df[["donor", "catpat", "clone_handle", "total_clone_count", "type_x_clone_count"]].drop_duplicates()

In [ ]:
def expansion_sweep_df_donor2(sdata_temp, catpat="MS", max_expansion=25):

    all_results = []

    for x in range(0, max_expansion + 1):

        # Get relevant columns
        obs_df = sdata_temp["gex"].obs[
             ["catpat", "donor", "Compartment", "clone_handle", "type_x_clone_count", "total_clone_count"] ## By dropping duplicates each clone will only be present once. 
        ].drop_duplicates().copy()

        # Subset on patient group and expansion threshold
        obs_df = obs_df[~obs_df["clone_handle"].isna()]
        obs_df = obs_df[obs_df["catpat"] == catpat]
        obs_df = obs_df[obs_df["total_clone_count"] > x]

        #Turn all categories into strings
        for col in obs_df.select_dtypes(["category"]):
            obs_df[col] = obs_df[col].cat.remove_unused_categories()

        ## Calculate a sum of type_x_clone_count
        obs_df = obs_df.groupby(["catpat", "donor", "Compartment"], observed=True, as_index=False)["type_x_clone_count"].sum()
        # Pivot to determine CSF/PB presence
        obs_df = obs_df.pivot_table(
            index=["donor"],
            columns="Compartment",
            values="type_x_clone_count",
            fill_value=0
        )

        # Calculate %CSF
        obs_df = obs_df.reset_index()
        obs_df["total"] = obs_df["CSF"] + obs_df["PB"]
        obs_df["CSF_pct"] = 100*obs_df["CSF"] / obs_df["total"]

        # Add expansion threshold
        obs_df["expansion_threshold"] = x

        # Add patient category
        obs_df["catpat"] = catpat

        all_results.append(obs_df)

    return pd.concat(all_results, ignore_index=True)

In [ ]:
## Make the dataframes
df_MS = expansion_sweep_df_donor2(sdata_temp, "MS")
df_ctrl = expansion_sweep_df_donor2(sdata_temp, "NIC")

## Concatenate
out_concat = pd.concat([df_MS, df_ctrl])

In [ ]:
plot_df = out_concat[out_concat["expansion_threshold"].isin([0, 5, 10, 15, 20])]

## ---- Correct plot_df by introducing zero values per donor x source ---- 
plot_df_backup = plot_df.copy()

# Complete donor × source combinations
all_sources = plot_df["expansion_threshold"].unique()

# Complete donor × source combinations
plot_df = (
    plot_df.set_index(["donor", "expansion_threshold"])
      .reindex(
          pd.MultiIndex.from_product(
              [plot_df["donor"].unique(), all_sources],
              names=["donor", "expansion_threshold"]
          )
      )
      .reset_index()
)

# Missing cell counts = 0
plot_df["CSF_pct"] = plot_df["CSF_pct"].fillna(0)

# Remove catpat
plot_df = plot_df.drop(columns = ["catpat"])

## Shift with 0.1 to avoid 0 values
plot_df["CSF_pct"] = plot_df["CSF_pct"] + 0.1

#Restore catpat
donor_info = plot_df_backup[["donor", "catpat"]].drop_duplicates("donor")
plot_df = plot_df.merge(donor_info, on="donor", how="left")


In [ ]:
# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(6.5,5))

# Boxplot (keep white boxes)
sb.boxplot(
    data=plot_df,
    x="expansion_threshold",
    y="CSF_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=plot_df,
    x="expansion_threshold",
    y="CSF_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.axhline(y=0.1, color="black")
ax.set_xlabel("Expansion thresholds (n cells > x)")
ax.set_ylabel("% of cells in CSF per donor") #% of cells in CSF\nnorm. to clone sizes

sb.despine(ax=ax)
plt.tight_layout()

fig.savefig(
    "../../05_plots/13_final_plots/13_boxplots_catexp_thresholds_percentage_cells.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()

#Save a list of expansion thresholds
threshold_order = [0, 5, 10, 15, 20]

# Initialize list to collect results
stats_list = []

for xtick in threshold_order:
    data_xtick = plot_df[plot_df["expansion_threshold"] == xtick]
    
    MS_vals = data_xtick[data_xtick["catpat"] == "MS"]["CSF_pct"]
    ctrl_vals = data_xtick[data_xtick["catpat"] == "NIC"]["CSF_pct"]
    
    if len(MS_vals) > 0 and len(ctrl_vals) > 0:
        stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
        
        stats_list.append({
            "expansion_threshold": xtick,
            "statistic": stat,
            "p_value": pval,
            "median_MS": MS_vals.median(),
            "median_NIC": ctrl_vals.median(),
            "n_MS": len(MS_vals),
            "n_NIC": len(ctrl_vals)
        })

# Convert to dataframe
df_stats = pd.DataFrame(stats_list)

# Apply Benjamini-Hochberg correction
reject, pvals_bh, _, _ = multipletests(
    df_stats["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

# Add results to dataframe
df_stats["p_value_BH"] = pvals_bh
df_stats["significant_BH"] = reject

df_stats

# Plot 4: Permutation of clonotypes - statistical thresholds

In [ ]:
obs_df = sdata["gex"].obs[["donor", "catpat", "clone_handle", "Compartment"]]

#Filter - exclude Pt4 and Pt5
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

#calculate the number of cells in CSF for all appropriate donors first (no catpat subset)
compart_sum = obs_df.groupby(["Compartment"]).size().reset_index(name="count")
CSF_cells = compart_sum.loc[compart_sum["Compartment"] == "CSF", "count"].iloc[0]
PB_cells = compart_sum.loc[compart_sum["Compartment"] == "PB", "count"].iloc[0]

#Calculate the fraction of cells in CSF of the data
CSF_pct = CSF_cells / (CSF_cells + PB_cells)
CSF_pct

#For grouped results
compart_sum_grp = obs_df.groupby(["Compartment", "catpat"]).size().reset_index(name="count").pivot(index = "catpat", columns="Compartment", values="count")
compart_sum_grp["total"] = compart_sum_grp["CSF"] + compart_sum_grp["PB"]  
compart_sum_grp["CSF.pct"] = compart_sum_grp["CSF"] / compart_sum_grp["total"] 
compart_sum_grp = compart_sum_grp.reset_index()

In [ ]:
## Given these statistics, make permutations to indicate what is highly significant
# Parameters
p_csf = CSF_pct  # compart_sum.loc[compart_sum["catpat"] == "RRMS", "CSF.pct"].iloc[0]
alpha = 0.05
max_cells = 80

rows = []

for n in range(1, max_cells + 1):
    for k in range(0, n + 1):
        purity = k / n

        # Binomial test under null
        pval = binomtest(k, n, p_csf, alternative="two-sided").pvalue

        if pval < alpha:
            rows.append({
                "n_cells": n,
                "purity": purity
            })

sig_df = pd.DataFrame(rows)

In [ ]:
# Prepare the figure
fig, ax = plt.subplots(figsize=(4, 3.75))

# Plot data points
ax.scatter(
    sig_df["n_cells"],
    sig_df["purity"],
    s=10,
    color="black"
)

ax.axhline(0.8333, linestyle="--", linewidth=1)
ax.axvline(6, linestyle="--", linewidth=1)

ax.set_xlabel("Number of cells discovered")
ax.set_ylabel("Fraction of cells\nin CSF of a given clone")
ax.set_title("Aggregated MS & Ctrl")

fig.tight_layout()

# ---- Save BEFORE show ----
fig.savefig(
    "../../05_plots/16_CD4analysis/5_Permutation_clone_size_vs_csf_fraction.pdf",
    bbox_inches="tight",
    format="pdf",
    dpi=400
)

plt.show()

In [ ]:
sig_df[sig_df["n_cells"] == 6]

# Plot 5: % of CSF-biased clones

In [ ]:
## To get to CSF-biased thresholds take everythin above 0.2
### 
csf_thresholds = sig_df[sig_df["purity"]>0.4].groupby("n_cells").min().reset_index()
csf_thresholds = csf_thresholds.rename(columns={"n_cells" : "total_clone_count", "purity" : "csf_threshold"})
csf_thresholds.head(5)

In [ ]:
## To get to PB-biased thresholds take everythin below 0.2
### 
pb_thresholds = sig_df[sig_df["purity"]<0.4].groupby("n_cells").max().reset_index()
pb_thresholds = pb_thresholds.rename(columns={"n_cells" : "total_clone_count", "purity" : "pb_threshold"})
pb_thresholds.tail(5)

#### Subgoal 1: Calculate global percentage of clones that are CSF-biased

In [ ]:
## Get obs_df
obs_df = sdata["gex"].obs
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[obs_df["clone_handle"].notna()]

## Select columns
obs_df = obs_df[["donor", "catpat", "clone_handle", "Compartment", "type_x_clone_count", "total_clone_count"]]

# Remove all unused categories
for col in obs_df.select_dtypes(include="category").columns:
    obs_df[col] = obs_df[col].cat.remove_unused_categories()

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Compute Compartment Percentages
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Compute % contribution of each compartment per clone
obs_df["Compartment_pct"] = 100 * obs_df["type_x_clone_count"] / obs_df["total_clone_count"]

# Create input dataframe for pivoting
df_input = obs_df[["donor", "clone_handle", "total_clone_count", "Compartment", "Compartment_pct", "catpat"]].drop_duplicates()

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Add flexible statistical thresholds
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# ## Add csf_threshold and pb_threshold
# df_input = df_input.merge(csf_thresholds).merge(pb_thresholds)

# ### Problem(fixed): Compartment = PB can be 100% for the Compartment-pct, but what we want is % CSF instead to be able to use the pb_thresholds that are below 0.2 to be significantly pb biased
# ### Where compartment == PB    modify the compartment pct to be a csf_percentage
# df_input["csf_comp_pct"] = np.where(df_input["Compartment"] == "PB", 100-df_input["Compartment_pct"], df_input["Compartment_pct"])

# ## Prune the columns
# df_input = df_input[["donor", "Compartment", "clone_handle", "total_clone_count", "csf_comp_pct", "catpat", "csf_threshold", "pb_threshold"]].drop_duplicates()

# ## Define lists of CSF-biased and PB-biased clones separately
# csf_list = df_input[(df_input["Compartment"] == "CSF") & (df_input["csf_comp_pct"] >= 100*df_input["csf_threshold"])]["clone_handle"].tolist()
# pb_list = df_input[(df_input["Compartment"] == "PB") & (df_input["csf_comp_pct"] <= 100*df_input["pb_threshold"])]["clone_handle"].tolist()

# ## annotate biased_ann
# df_input["biased_ann"] = np.where(df_input["clone_handle"].isin(csf_list), "csf_biased", 
                                  # np.where(df_input["clone_handle"].isin(pb_list), "pb_biased", "neutral"))

## Prune columns / drop duplicates
# df_input = df_input[["donor", "catpat", "clone_handle", "total_clone_count", "csf_comp_pct",  "csf_threshold", "pb_threshold", "biased_ann"]].drop_duplicates()

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Add static statistical thresholds
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
df_input["biased_ann"] = np.where((df_input["Compartment"]== "CSF") & (df_input["Compartment_pct"] >= 83.333), "csf_biased", 
                                  np.where((df_input["Compartment"]== "PB") & (df_input["Compartment_pct"] == 100), "pb_biased", "neutral"))

## Prune columns / drop duplicates
df_input = df_input[["donor", "catpat", "clone_handle", "total_clone_count",  "biased_ann"]].drop_duplicates()


# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# # Enumerate per donor how many CSF-biased clones each donor has
# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
count_df = df_input[["donor", "catpat", "biased_ann"]].value_counts().reset_index()
count_df = count_df.pivot_table(index = ["donor", "catpat"], columns = "biased_ann", values = "count", fill_value=0).reset_index()
count_df["csfbiased_pct"] = 100*count_df["csf_biased"] / (count_df["csf_biased"]+count_df["neutral"]+count_df["pb_biased"])
count_df

#### Subgoal 2: Plot % of clones that are CSF-biased per donor

In [ ]:
# Define manual colors
palette = {
    "NIC": "#FFFFFF",
    "MS": "#000000"
}

fig, ax = plt.subplots(figsize=(3, 4))

# Boxplot
sb.boxplot(
    data=count_df,
    x="catpat",
    y="csfbiased_pct",
    order=["NIC", "MS"],
    width=0.5,
    showcaps=True,
    showfliers=False,
    boxprops={"facecolor": "white", "edgecolor": "black"},
    whiskerprops={"color": "black"},
    capprops={"color": "black"},
    medianprops={"color": "black"},
    ax=ax
)

# Stripplot
sb.stripplot(
    data=count_df,
    x="catpat",
    y="csfbiased_pct",
    order=["NIC", "MS"],
    hue="catpat",
    hue_order=["NIC", "MS"],
    palette=palette,
    dodge=False,
    size=8,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("CSF-biased (%)")
ax.set_ylim(-5, 110)
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.show()

#### Over different expansion thresholds

In [ ]:
## Prune columns / drop duplicates
# df_input = df_input[["donor", "catpat", "clone_handle", "total_clone_count", "csf_comp_pct",  "csf_threshold", "pb_threshold", "biased_ann"]].drop_duplicates()

# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# # Enumerate per donor how many CSF-biased clones each donor has over different expansion thresholds
# # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
count_df = df_input[["donor", "catpat", "biased_ann", "total_clone_count"]].value_counts().reset_index()
count_df = count_df.pivot_table(index = ["donor", "catpat", "total_clone_count"], columns = "biased_ann", values = "count", fill_value=0).reset_index()

## Make 5 dfs
df0 = count_df[count_df["total_clone_count"] > 0].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df0["source"] = "0"

df5 = count_df[count_df["total_clone_count"] > 5].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df5["source"] = "5"

df10 = count_df[count_df["total_clone_count"] > 10].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df10["source"] = "10"

df15 = count_df[count_df["total_clone_count"] > 15].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df15["source"] = "15"

df20 = count_df[count_df["total_clone_count"] > 20].groupby(["donor", "catpat"], observed=True)[["csf_biased", "neutral", "pb_biased"]].sum().reset_index()
df20["source"] = "20"

### sum_df
sum_df = pd.concat([df0, df5, df10, df15, df20])

### frequency of csf biased clones
sum_df["csfbiased_pct"] = 100*sum_df["csf_biased"] / (sum_df["csf_biased"]+sum_df["neutral"]+sum_df["pb_biased"])

## ---- Correct plot_df by introducing zero values per donor x source ---- 
sum_df_backup = sum_df.copy()

# Complete donor × source combinations
all_sources = sum_df["source"].unique()

# Complete donor × source combinations
sum_df = (
    sum_df.set_index(["donor", "source"])
      .reindex(
          pd.MultiIndex.from_product(
              [sum_df["donor"].unique(), all_sources],
              names=["donor", "source"]
          )
      )
      .reset_index()
)

# Missing cell counts = 0
sum_df["csfbiased_pct"] = sum_df["csfbiased_pct"].fillna(0)

# Remove catpat
sum_df = sum_df.drop(columns = ["catpat"])

## Shift with 0.1 to avoid 0 values
# sum_df["csfbiased_pct"] = sum_df["csfbiased_pct"] + 0.1

#Restore catpat
donor_info = sum_df_backup[["donor", "catpat"]].drop_duplicates("donor")
sum_df = sum_df.merge(donor_info, on="donor", how="left")


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
#             plot
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(5,4))

# Boxplot (keep white boxes)
sb.boxplot(
    data=sum_df,
    x="source",
    y="csfbiased_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    # log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=sum_df,
    x="source",
    y="csfbiased_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    # log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.axhline(y=0)
ax.set_xlabel("Expansion thresholds (n cells > x)")
ax.set_ylabel("% CSF-biased clones") #% of cells in CSF\nnorm. to clone sizes

sb.despine(ax=ax)
ax.grid(False)
plt.tight_layout()

fig.savefig(
    "../../05_plots/13_final_plots/XX_CD4_boxplot_freq_of_CSF_biased_clones.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()

#Save a list of expansion thresholds
threshold_order = ["0", "5", "10", "15", "20"]

# Initialize list to collect results
stats_list = []

for xtick in threshold_order:
    data_xtick = sum_df[sum_df["source"] == xtick]
    
    MS_vals = data_xtick[data_xtick["catpat"] == "MS"]["csfbiased_pct"]
    ctrl_vals = data_xtick[data_xtick["catpat"] == "NIC"]["csfbiased_pct"]
    
    if len(MS_vals) > 0 and len(ctrl_vals) > 0:
        stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
        
        stats_list.append({
            "expansion_threshold": xtick,
            "statistic": stat,
            "p_value": pval,
            "median_MS": MS_vals.median(),
            "median_NIC": ctrl_vals.median(),
            "n_MS": len(MS_vals),
            "n_NIC": len(ctrl_vals)
        })

# Convert to dataframe
df_stats = pd.DataFrame(stats_list)

# Apply Benjamini-Hochberg correction
reject, pvals_bh, _, _ = multipletests(
    df_stats["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

# Add results to dataframe
df_stats["p_value_BH"] = pvals_bh
df_stats["significant_BH"] = reject

df_stats

# Plot 6: Clonal sharing per cluster

### PB clusters

In [ ]:
## Reproduce Kavaka et al analysis with clonal overlap among each cluster
#### Of all the CSF clonotypes - which cluster contains the most overlap?

obs_df = sdata["gex"].obs[["donor", "Compartment", "catpat", "leiden_1.8", "clone_handle"]].copy()
obs_df = obs_df[obs_df["catpat"] != "IC"]
obs_df = obs_df[~obs_df["clone_handle"].isna()]

## TCSF cells
obs_csf = obs_df[obs_df["Compartment"]=="CSF"]
list_csf_clones = obs_csf["clone_handle"].drop_duplicates().tolist()

# Ensure CSF clones is a set
csf_clones = set(list_csf_clones)

# Define the clusters to iterate over and the cluster column. 
# These clusters has to be PB clusters
clusters =  ["0", "5", "6", "4", "8", "15", "10", "3", "9", "17", "18", "13", "16", "20"]
cluster_col = "leiden_1.8"

# Define an empty dataframe
results = []

for cl in clusters:
    # Subset for given cluster in PB
    sub = obs_df[
        (obs_df[cluster_col] == cl) &
        (obs_df["Compartment"] == "PB")
    ]

    cluster_clones = set(sub["clone_handle"].drop_duplicates())

    # Overlap and union
    overlap = csf_clones & cluster_clones
    union = csf_clones | cluster_clones

    pct_csf_in_cluster = (
        len(overlap) / len(csf_clones) * 100
        if len(csf_clones) > 0 else np.nan
    )

    jaccard = (
        len(overlap) / len(union) * 100
        if len(union) > 0 else np.nan
    )

    results.append({
        cluster_col: cl,
        "n_csf_clones": len(csf_clones),
        "n_cluster_clones": len(cluster_clones),
        "n_overlap": len(overlap),
        "pct_csf_in_cluster": pct_csf_in_cluster,
        "jaccard_index": jaccard
    })

df_results = pd.DataFrame(results)
df_results



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot 1: % CSF in cluster ---
sb.barplot(
    data=df_results,
    x="leiden_1.8",
    y="pct_csf_in_cluster",
    # order=order,
    ax=axes[0]
)

axes[0].set_title("% CSF clones in cluster")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Percentage")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(False)

# --- Plot 2: Jaccard index ---
sb.barplot(
    data=df_results,
    x="leiden_1.8",
    y="jaccard_index",
    # order=order,
    ax=axes[1]
)

axes[1].set_title("Jaccard index")
axes[1].set_xlabel("PB cluster")
axes[1].set_ylabel("Percentage")
axes[1].tick_params(axis="x", rotation=0)
axes[1].grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/16_CD4analysis/22_CD4_Jaccard_overlap_leiden_1.8.pdf")
plt.show()

### CSF clusters

In [ ]:
obs_df = sdata["gex"].obs[["donor", "Compartment", "catpat", "leiden_1.8", "clone_handle"]].copy()
obs_df = obs_df[~obs_df["clone_handle"].isna()]

## PB cells
obs_pb = obs_df[obs_df["Compartment"]=="PB"]
list_pb_clones = obs_pb["clone_handle"].drop_duplicates().tolist()

# Ensure pb clones is a set
pb_clones = set(list_pb_clones)

# Define the clusters to iterate over and the cluster column. 
# These clusters has to be PB clusters
clusters =  ["1", "2", "7", "11", "12", "14", "19"]
cluster_col = "leiden_1.8"

# Define an empty dataframe
results = []

for cl in clusters:
    # Subset for given cluster in CSF
    sub = obs_df[
        (obs_df[cluster_col] == cl) &
        (obs_df["Compartment"] == "CSF")
    ]

    cluster_clones = set(sub["clone_handle"].drop_duplicates())

    # Overlap and union
    overlap = pb_clones & cluster_clones
    union = pb_clones | cluster_clones

    pct_pb_in_cluster = (
        len(overlap) / len(pb_clones) * 100
        if len(pb_clones) > 0 else np.nan
    )

    jaccard = (
        len(overlap) / len(union) * 100
        if len(union) > 0 else np.nan
    )

    results.append({
        cluster_col: cl,
        "n_pb_clones": len(pb_clones),
        "n_cluster_clones": len(cluster_clones),
        "n_overlap": len(overlap),
        "pct_pb_in_cluster": pct_pb_in_cluster,
        "jaccard_index": jaccard
    })

df_results = pd.DataFrame(results)
df_results



In [ ]:
## Make a column plot that describes clonal sharing amongst CD8 T cells
# Ensure correct ordering
# order = ["TN", "TSCM", "TCM", "TEM", "TE", "TAct"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot 1: % PB in cluster ---
sb.barplot(
    data=df_results,
    x="leiden_1.8",
    y="pct_pb_in_cluster",
    # order=order,
    ax=axes[0]
)

axes[0].set_title("% PB clones in cluster")
axes[0].set_ylim(0, 1.9)
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Percentage")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(False)

# --- Plot 2: Jaccard index ---
sb.barplot(
    data=df_results,
    x="leiden_1.8",
    y="jaccard_index",
    # order=order,
    ax=axes[1]
)

axes[1].set_title("Jaccard index")
axes[1].set_xlabel("PB cluster")
axes[1].set_ylabel("Percentage")
axes[1].tick_params(axis="x", rotation=0)
axes[1].grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/16_CD4analysis/22_CSF_clustersCD4_Jaccard_overlap_leiden_1.8.pdf")
plt.show()

# Plot 7: Column plots to visualize compartment-wise expansion

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Define set 5, i.e. all clone_handles in the data with expansion above 5
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

## Check whether total_clone_count fits with the number of rows
obs_df = sdata["gex"].obs.groupby(["clone_handle", "total_clone_count"], observed=True).size().reset_index(name="row_count")
## Number of rows where the clone count and row count are different should be 0
print("Number of rows where the clone_count is different from row count: ", len(obs_df[obs_df["total_clone_count"] != obs_df["row_count"]]))

## Now define set5
obs_df = sdata["gex"].obs.copy()
set5 = obs_df[obs_df["total_clone_count"] > 5]["clone_handle"].drop_duplicates().tolist()
print("Number of set5 CD4 clones: ", len(set5))

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 1: Data Preparation and Filtering
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Make a data copy and filter that
sdata_highfrequent = sdata.copy()

# Combined criteria:
comb_sets = set5

# Make clone_handle categorical
sdata_highfrequent["gex"].obs["clone_handle"] = sdata_highfrequent["gex"].obs["clone_handle"].astype("category")

# Filter for combined clone set & remove certain donors
mu.pp.filter_obs(sdata_highfrequent["gex"], "clone_handle", lambda x: (x.isin(comb_sets)))
mu.pp.filter_obs(sdata_highfrequent["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))

#Group annotation
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]

#Match the lists defined above to define "RRMS" and "Control"
sdata_highfrequent["gex"].obs["catpat"] = pd.Categorical(np.where(sdata_highfrequent["gex"].obs["donor"].isin(patient_list), "RRMS", np.where(sdata_highfrequent["gex"].obs["donor"].isin(control_list), "Control", "Error")))

#Print two reports
print("I have this many hyperexpanded clonotypes among RRMS patients: ", 
      len(sdata_highfrequent["gex"].obs[sdata_highfrequent["gex"].obs["catpat"] == "RRMS"]["clone_handle"].drop_duplicates()))

print("I have this many hyperexpanded clonotypes among Ctrl patients: ", 
      len(sdata_highfrequent["gex"].obs[sdata_highfrequent["gex"].obs["catpat"] == "Control"]["clone_handle"].drop_duplicates()))

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 2: UMAP Plot
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Visualize clones using UMAP projection
# 
# sc.pl.umap(sdata_highfrequent["gex"], color=["L3_ann"], legend_fontweight='bold', frameon=False, vmin=0, legend_fontoutline=2, size=10)
# 

# fig.savefig("..\\..\\05_plots\\08_AfterAntwerp\\12_umap_shared_and_PB_clonotypes.pdf",bbox_inches='tight', format="pdf", dpi=400)

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 3: Compute Compartment Percentages
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Compute % contribution of each compartment per clone
sdata_highfrequent["gex"].obs["Compartment_pct"] = 100 * sdata_highfrequent["gex"].obs["type_x_clone_count"] / sdata_highfrequent["gex"].obs["total_clone_count"]

# Sanity check: number of unique clones
print("I have this many unique clones: ", len(sdata_highfrequent["gex"].obs["clone_handle"].drop_duplicates()))

# Create input dataframe for pivoting
df_input = sdata_highfrequent["gex"].obs[["clone_handle", "Compartment", "Compartment_pct", "catpat"]]

# Pivot the data so compartments become columns
# pivot_table_comp = df_input.pivot_table(index=["clone_handle"], columns="Compartment", values="Compartment_pct", fill_value=0)
pivot_table_comp = df_input.pivot_table(index=["clone_handle", "catpat"], columns="Compartment", values="Compartment_pct", fill_value=0)

#Did pivot generate zero values? No
# pivot_table_comp[pivot_table_comp["CSF"] == pivot_table_comp["PB"]]

# Rename columns
pivot_table_comp.columns = [f"{col}.pct" for col in pivot_table_comp.columns]

# Sort to show CSF-high clones first
pivot_table_comp = pivot_table_comp.sort_values(by="CSF.pct", ascending=False)  #Stores the compartment-wise dataset for plotting
clone_order = pivot_table_comp.index.tolist() #Will be used for L3_ann plots later
num_bars_comp = len(pivot_table_comp)

#Clone order for PB and CSF individually - these indexes are combinations of clone_handle and catpat (!!): 
pivot_table_comp.sort_values(by="CSF.pct", ascending=False)
clone_order_pb = pivot_table_comp[pivot_table_comp["CSF.pct"] != 100].sort_values(by="CSF.pct", ascending=False).index.tolist()  #Stores the compartment-wise dataset for plotting
clone_order_csf = pivot_table_comp[pivot_table_comp["PB.pct"] != 100].sort_values(by="CSF.pct", ascending=False).index.tolist()  #Stores the compartment-wise dataset for plotting

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 4: Prepare L3_cluster data for PB
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Filter dataset to PB cells from selected clones and donors
sdata_highfrequent_pb = sdata.copy()
sdata_highfrequent_pb["gex"].obs["clone_handle"] = sdata_highfrequent_pb["gex"].obs["clone_handle"].astype("category")

#Match the lists defined above to define "RRMS" and "Control"
sdata_highfrequent_pb["gex"].obs["catpat"] = pd.Categorical(np.where(sdata_highfrequent_pb["gex"].obs["donor"].isin(patient_list), "RRMS", np.where(sdata_highfrequent_pb["gex"].obs["donor"].isin(control_list), "Control", "Error")))

   #Comb_sets were defined earlier in this code. 
mu.pp.filter_obs(sdata_highfrequent_pb["gex"], "clone_handle", lambda x: (x.isin(comb_sets)))
mu.pp.filter_obs(sdata_highfrequent_pb["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
mu.pp.filter_obs(sdata_highfrequent_pb["gex"], "Compartment", lambda x: (x == "PB")) 
sdata_highfrequent_pb.update()
sdata_highfrequent_pb["gex"].obs["clone_handle"] = sdata_highfrequent_pb["gex"].obs["clone_handle"].cat.remove_unused_categories()
sdata_highfrequent_pb["gex"].obs["Compartment"] = sdata_highfrequent_pb["gex"].obs["Compartment"].cat.remove_unused_categories()
sdata_highfrequent_pb["gex"].obs["L3_ann"] = sdata_highfrequent_pb["gex"].obs["L3_ann"].cat.remove_unused_categories()
sdata_highfrequent_pb.update()

# Group by clone_handle, L3_ann and catpat and count the number of cells in the single-cell object
L3_counts = (
    sdata_highfrequent_pb["gex"].obs
    .groupby(["clone_handle", "L3_ann", "catpat"])
    .size()
    .reset_index(name="count")
)

# Pivot count data to generate a wide format - this generates many rows full with 0 values. 
count_tableL3 = L3_counts[["clone_handle",  "catpat", "L3_ann", "count"]].drop_duplicates().pivot(index=["clone_handle", "catpat"], columns=["L3_ann"], values="count").fillna(0).reset_index()

# Remove rows where all cluster counts are zero
cluster_columns = count_tableL3.columns.difference(["clone_handle", "catpat"])
count_tableL3 = count_tableL3.loc[(count_tableL3[cluster_columns].sum(axis=1) > 0)]

# Add totals
count_tableL3["total"] = count_tableL3.select_dtypes(include="number").sum(axis=1)

# Calculate percentages for each numeric column except total: 
numeric_columns = count_tableL3.select_dtypes(include="number").columns.tolist()
for col in numeric_columns:
    if col != 'total':
        count_tableL3[col + "_pct"] = 100 * count_tableL3[col] / count_tableL3["total"]

### ChatGPT suggests melt --> pivot.. But we are not using this exact dataframe later anyway. Save brain power by simply picking the _pct columns pairing it would the identifier columns from earlier. 
# Select final wide-format data for plotting
# value_cols = [col for col in count_tableL3.columns if col.endswith("_pct")]
# pivot_table_L3 = count_tableL3[["clone_handle", "catpat"] + value_cols]

## Nevermind - this is necessary, because the identifier variables all needs to be stacked as an index, which matches clone_order_pb later. 
# Melt generates a long format dataframe
 ## Only use _pct columns. 
value_vars_pct = [col for col in count_tableL3.columns if col.endswith('_pct')]
count_tableL3_long = count_tableL3.melt(id_vars=["clone_handle", "catpat"], value_vars=value_vars_pct, var_name="L3_ann", value_name="Percentage")

# Pivot to wide format. pivot_table_L3 now stores the cluster-wise data for plotting
pivot_table_L3 = count_tableL3_long.pivot(index=["clone_handle", "catpat"], columns="L3_ann", values="Percentage").fillna(0)

pivot_table_L3 = pivot_table_L3.loc[clone_order_pb]  #Requires exact matches

# save the final _pb dataframe. 
pivot_table_L3_pb = pivot_table_L3.loc[:, pivot_table_L3.sum(axis=0) > 0]
num_bars_L3_pb = len(pivot_table_L3)

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 5: Prepare L3_cluster data for CSF
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Filter dataset to PB cells from selected clones and donors
sdata_highfrequent_csf = sdata.copy()
sdata_highfrequent_csf["gex"].obs["clone_handle"] = sdata_highfrequent_csf["gex"].obs["clone_handle"].astype("category")

#Match the lists defined above to define "RRMS" and "Control"
sdata_highfrequent_csf["gex"].obs["catpat"] = pd.Categorical(np.where(sdata_highfrequent_csf["gex"].obs["donor"].isin(patient_list), "RRMS", np.where(sdata_highfrequent_csf["gex"].obs["donor"].isin(control_list), "Control", "Error")))

#Comb_sets were defined earlier in this code. 
mu.pp.filter_obs(sdata_highfrequent_csf["gex"], "clone_handle", lambda x: (x.isin(comb_sets)))
mu.pp.filter_obs(sdata_highfrequent_csf["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
mu.pp.filter_obs(sdata_highfrequent_csf["gex"], "Compartment", lambda x: (x == "CSF")) 
sdata_highfrequent_csf.update()
sdata_highfrequent_csf["gex"].obs["clone_handle"] = sdata_highfrequent_csf["gex"].obs["clone_handle"].cat.remove_unused_categories()
sdata_highfrequent_csf["gex"].obs["Compartment"] = sdata_highfrequent_csf["gex"].obs["Compartment"].cat.remove_unused_categories()
sdata_highfrequent_csf["gex"].obs["L3_ann"] = sdata_highfrequent_csf["gex"].obs["L3_ann"].cat.remove_unused_categories()
sdata_highfrequent_csf.update()

# Group by clone_handle, L3_ann and catpat and count the number of cells in the single-cell object
L3_counts = (
    sdata_highfrequent_csf["gex"].obs
    .groupby(["clone_handle", "L3_ann", "catpat"])
    .size()
    .reset_index(name="count")
)

# Pivot count data to generate a wide format - this generates many rows full with 0 values. 
count_tableL3 = L3_counts[["clone_handle",  "catpat", "L3_ann", "count"]].drop_duplicates().pivot(index=["clone_handle", "catpat"], columns=["L3_ann"], values="count").fillna(0).reset_index()

# Remove rows where all cluster counts are zero
cluster_columns = count_tableL3.columns.difference(["clone_handle", "catpat"])
count_tableL3 = count_tableL3.loc[(count_tableL3[cluster_columns].sum(axis=1) > 0)]

# Add totals
count_tableL3["total"] = count_tableL3.select_dtypes(include="number").sum(axis=1)

# Calculate percentages for each numeric column except total: 
numeric_columns = count_tableL3.select_dtypes(include="number").columns.tolist()
for col in numeric_columns:
    if col != 'total':
        count_tableL3[col + "_pct"] = 100 * count_tableL3[col] / count_tableL3["total"]

### ChatGPT suggests melt --> pivot.. But we are not using this exact dataframe later anyway. Save brain power by simply picking the _pct columns pairing it would the identifier columns from earlier. 
# Select final wide-format data for plotting
# value_cols = [col for col in count_tableL3.columns if col.endswith("_pct")]
# pivot_table_L3 = count_tableL3[["clone_handle", "catpat"] + value_cols]

## Nevermind - this is necessary, because the identifier variables all needs to be stacked as an index, which matches clone_order_csf later.
# Melt generates a long format dataframe
 ## Only use _pct columns. 
value_vars_pct = [col for col in count_tableL3.columns if col.endswith('_pct')]
count_tableL3_long = count_tableL3.melt(id_vars=["clone_handle", "catpat"], value_vars=value_vars_pct, var_name="L3_ann", value_name="Percentage")

# Pivot to wide format. pivot_table_L3 now stores the cluster-wise data for plotting
pivot_table_L3 = count_tableL3_long.pivot(index=["clone_handle", "catpat"], columns="L3_ann", values="Percentage").fillna(0)

pivot_table_L3 = pivot_table_L3.loc[clone_order_csf]  #Requires exact matches

# save the final _csf dataframe. 
pivot_table_L3_csf = pivot_table_L3.loc[:, pivot_table_L3.sum(axis=0) > 0]
num_bars_L3_csf = len(pivot_table_L3)

# %%%%%%%%%%%%%%%%%%%%%
# --- Figure 1: Full plot with legend - CSF to PB bias - with facet---
# %%%%%%%%%%%%%%%%%%%%%

#SET TO DEFAULT SEABORN STYLE
sb.reset_orig()

#Reset index to unstack the multiindex
pivot_table_comp = pivot_table_comp.reset_index()

# Prepare x positions and bar width
x = np.arange(len(pivot_table_comp))
width = 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True, constrained_layout=True)
width = 1

# Split by group
pivot_ms = pivot_table_comp[pivot_table_comp["catpat"] == "RRMS"]
pivot_control = pivot_table_comp[pivot_table_comp["catpat"] == "Control"]

# Plot MS
x_ms = np.arange(len(pivot_ms))
ax1.bar(x_ms, pivot_ms["CSF.pct"].values, width, label="CSF", color="#4B8AC1")
ax1.bar(x_ms, pivot_ms["PB.pct"].values, width, bottom=pivot_ms["CSF.pct"].values, label="PB", color="#C06A6A")
ax1.set_title("RRMS")
ax1.set_ylabel("Percentage")
ax1.set_xticks([])
ax1.axvline(-0.5, color="black")
ax1.axhline(83.3333, color="black")

# Plot Control
x_control = np.arange(len(pivot_control))
ax2.bar(x_control, pivot_control["CSF.pct"].values, width, label="CSF", color="#4B8AC1")
ax2.bar(x_control, pivot_control["PB.pct"].values, width, bottom=pivot_control["CSF.pct"].values, label="PB", color="#C06A6A")
ax2.set_title("Control")
ax2.set_xticks([])
ax2.axvline(-0.5, color="black")
ax2.axhline(83.3333, color="black")

# Clean up spines
for ax in [ax1, ax2]:
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

# Legend and title
fig.legend(title="Compartment", bbox_to_anchor=(1.05, 1), loc="upper left")
fig.suptitle("CSF vs PB composition per hyperexpanded clone (n > 5 cells)", fontsize=14)

fig.savefig("../../05_plots/16_CD4analysis/6_CD4_PB_to_CSF_transition_expansion_FULL.pdf", bbox_inches='tight', dpi=400)

# %%%%%%%%%%%%%%%%%%%%%
# --- Figure 2: FUll plot of L3_ann clusters from CSF to PB bias ---
# %%%%%%%%%%%%%%%%%%%%%


#reset index to unstack the multiindex: 
pivot_table_L3_pb_plot = pivot_table_L3_pb.reset_index()

# Set up colors
l3_labels_pb = sdata["gex"].obs["L3_ann"].cat.categories
l3_colors_pb = sdata["gex"].uns["leiden_1.8_colors"]
l3_color_dict_pb = dict(zip(l3_labels_pb, l3_colors_pb))

# Create one subplot per category (MS, Control)
catpats = ["RRMS", "Control"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(8 * n_facets, 6), constrained_layout=True, sharey=True)

if n_facets == 1:
    axes = [axes]  # Ensure it's always a list

# Plot each group
for i, cat in enumerate(catpats):
    ax = axes[i]
    
    df_cat = pivot_table_L3_pb_plot[pivot_table_L3_pb_plot["catpat"] == cat].reset_index(drop=True)
    x = np.arange(len(df_cat))
    bottom_vals = np.zeros(len(df_cat))
    
    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        color = l3_color_dict_pb.get(l3_cluster.replace("_pct", ""), "#333333")
        y_vals = df_cat[l3_cluster].values
        ax.bar(x, y_vals, width=1, bottom=bottom_vals, label=l3_cluster.replace("_pct", ""), color=color)
        bottom_vals += y_vals

    ax.set_title(f"{cat} clones\nL3 cluster contributions ({len(df_cat)} clones)", fontsize=12)
    ax.set_xlabel("Clone Handle")
    if i == 0:
        ax.set_ylabel("Percentage of PB cells")
    else:
        ax.set_ylabel("")
    ax.set_xticks([])
    ax.axvline(-0.5, color="black", linewidth=1)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

# Add legend to the last axis
axes[-1].legend(title="L3 annotations", bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)

# Save the figure
fig.savefig("../../05_plots/16_CD4analysis/7_CD4_PB_phenotypes_FULL.pdf", bbox_inches='tight', format="pdf", dpi=400)
plt.show()

# %%%%%%%%%%%%%%%%%%%%%
# --- Figure 5:  FUll plot of L3_ann clusters for CSF from CSF to PB bias ---
# %%%%%%%%%%%%%%%%%%%%%

# Ensure catpat is accessible
pivot_table_L3_csf_plot = pivot_table_L3_csf.reset_index()

# Extract L3_ann colors
l3_labels_csf = sdata["gex"].obs["L3_ann"].cat.categories
l3_colors_csf = sdata["gex"].uns["leiden_1.8_colors"]
l3_color_dict_csf = dict(zip(l3_labels_csf, l3_colors_csf))

# Faceting setup
catpats = ["RRMS", "Control"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(8 * n_facets, 6), constrained_layout=True, sharey=True)
if n_facets == 1:
    axes = [axes]  # Ensure iterable

# Plot each facet
for i, cat in enumerate(catpats):
    ax = axes[i]
    df_cat = pivot_table_L3_csf_plot[pivot_table_L3_csf_plot["catpat"] == cat].reset_index(drop=True)
    x = np.arange(len(df_cat))
    width = 1
    bottom_vals = np.zeros(len(df_cat))

    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        cluster_label = l3_cluster.replace("_pct", "")
        y_vals = df_cat[l3_cluster].values
        color = l3_color_dict_csf.get(cluster_label, "#333333")
        ax.bar(x, y_vals, width, bottom=bottom_vals, color=color, label=cluster_label)
        bottom_vals += y_vals

    ax.set_title(f"{cat} clones\nL3 cluster contributions ({len(df_cat)} clones)", fontsize=12)
    ax.set_xlabel("Clone Handle")
    if i == 0:
        ax.set_ylabel("Percentage of CSF cells")
    else:
        ax.set_ylabel("")

    ax.set_xticks([])
    ax.axvline(-0.5, color="black", linewidth=1)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

# Legend on the last subplot
axes[-1].legend(title="L3 annotations", bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)

# Save the faceted figure
fig.savefig("../../05_plots/16_CD4analysis/8_CD8_CSF_column_L3_ann_trends_FULL.pdf", bbox_inches='tight', format="pdf", dpi=400)
plt.show()

# %%%%%%%%%%%
# --- Figure : Naked CSF to PB bias ---
# %%%%%%%%%%%

# Ensure index is reset
pivot_table_comp = pivot_table_comp.reset_index()

# Setup
catpats = ["RRMS", "Control"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)

if n_facets == 1:
    axes = [axes]  # Ensure list in single facet case

# Plot per facet
for i, cat in enumerate(catpats):
    ax = axes[i]
    df_cat = pivot_table_comp[pivot_table_comp["catpat"] == cat].reset_index(drop=True)
    
    #Add a horizontal line that represent the mean %CSF of cells
    ax.axhline(y=83.33, color='black', linestyle='solid', linewidth=0.5)

    x = np.arange(len(df_cat))
    width = 1

    ax.bar(x, df_cat["CSF.pct"].values, width, color="#4B8AC1")
    ax.bar(x, df_cat["PB.pct"].values, width, bottom=df_cat["CSF.pct"].values, color="#C06A6A")
    
    # Clean up styling
    ax.set_xticks([])
    ax.tick_params(axis='y', labelsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(f"{cat} clones")

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    ax.axvline(-0.5, color="black", linewidth=1)

# Save figure
fig.savefig("../../05_plots/16_CD4analysis/9_CD4_PB_to_CSF_percentage_transition_hyperexpanded.pdf", bbox_inches='tight', format="pdf", dpi=400)
plt.show()

# %%%%%%%%%%%%%%%%%%%%%
# --- Figure : NAKED plot of L3_ann clusters from PB ---
# %%%%%%%%%%%%%%%%%%%%%

#reset index to unstack the multiindex: 
pivot_table_L3_pb_plot = pivot_table_L3_pb.reset_index()

# Set up colors
l3_labels_pb = sdata["gex"].obs["L3_ann"].cat.categories
l3_colors_pb = sdata["gex"].uns["leiden_1.8_colors"]
l3_color_dict_pb = dict(zip(l3_labels_pb, l3_colors_pb))

# Create one subplot per category (MS, Control)
catpats = ["RRMS", "Control"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)

if n_facets == 1:
    axes = [axes]  # Ensure it's always a list

# Plot each group
for i, cat in enumerate(catpats):
    ax = axes[i]
    
    df_cat = pivot_table_L3_pb_plot[pivot_table_L3_pb_plot["catpat"] == cat].reset_index(drop=True)
    x = np.arange(len(df_cat))
    bottom_vals = np.zeros(len(df_cat))
    
    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        color = l3_color_dict_pb.get(l3_cluster.replace("_pct", ""), "#333333")
        y_vals = df_cat[l3_cluster].values
        ax.bar(x, y_vals, width=1, bottom=bottom_vals, label=l3_cluster.replace("_pct", ""), color=color)
        bottom_vals += y_vals

    ax.set_title(f"{cat} clones")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    # ax.set_yticks([])
    ax.tick_params(axis='y', labelsize=9)
    ax.axvline(-0.5, color="black", linewidth=1)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

# Save the figure
fig.savefig("../../05_plots/16_CD4analysis/10_CD4_PB_LOESS_trends_by_L3_cluster_facet_catpat_NAKED.pdf", bbox_inches='tight', format="pdf", dpi=400)
plt.show()

# %%%%%%%%%%%%%%%%%%%%%
# --- Figure : NAKED plot of L3_ann clusters from CSF ---
# %%%%%%%%%%%%%%%%%%%%%

#reset index to unstack the multiindex: 
pivot_table_L3_csf_plot = pivot_table_L3_csf.reset_index()

# Set up colors
l3_labels_csf = sdata["gex"].obs["L3_ann"].cat.categories
l3_colors_csf = sdata["gex"].uns["leiden_1.8_colors"]
l3_color_dict_csf = dict(zip(l3_labels_csf, l3_colors_csf))

# Create one subplot per category (MS, Control)
catpats = ["RRMS", "Control"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)

if n_facets == 1:
    axes = [axes]  # Ensure it's always a list

# Plot each group
for i, cat in enumerate(catpats):
    ax = axes[i]
    
    df_cat = pivot_table_L3_csf_plot[pivot_table_L3_csf_plot["catpat"] == cat].reset_index(drop=True)
    x = np.arange(len(df_cat))
    bottom_vals = np.zeros(len(df_cat))
    
    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        color = l3_color_dict_csf.get(l3_cluster.replace("_pct", ""), "#333333")
        y_vals = df_cat[l3_cluster].values
        ax.bar(x, y_vals, width=1, bottom=bottom_vals, label=l3_cluster.replace("_pct", ""), color=color)
        bottom_vals += y_vals

    ax.set_title(f"{cat} clones")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    # ax.set_yticks([])
    ax.tick_params(axis='y', labelsize=9)
    ax.axvline(-0.5, color="black", linewidth=1)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

# Save the figure
fig.savefig("../../05_plots/16_CD4analysis/11_CD4_CSF_phenotypes_L3_cluster_facet_NAKED_catpat.pdf", bbox_inches='tight', format="pdf", dpi=400)
plt.show()


# %%%%%%%%%%%%%%%%%%%%%
# --- Figure 7:  LOESS regression of PB ---
# %%%%%%%%%%%%%%%%%%%%%

# Faceting setup
catpats = ["RRMS", "Control"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)
if n_facets == 1:
    axes = [axes]

for i, cat in enumerate(catpats):
    ax = axes[i]
    
    # Subset data
    df_cat = pivot_table_L3_pb_plot[pivot_table_L3_pb_plot["catpat"] == cat].reset_index(drop=True)
    x_vals = np.arange(len(df_cat))

    # LOWESS plot per L3 cluster
    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        cluster_label = l3_cluster.replace("_pct", "")
        y_vals = df_cat[l3_cluster].values
        smoothed = lowess(y_vals, x_vals, frac=0.35)
        color = l3_color_dict_pb.get(cluster_label, "#333333")
        ax.plot(smoothed[:, 0], smoothed[:, 1], label=cluster_label, color=color, linewidth=2)

    # Axes styling
    ax.set_xticks([])
    # ax.set_yticks([0, 20, 40, 60, 80])
    ax.set_ylim(0, 100)
    ax.set_xlabel("")
    ax.set_title(f"{cat} clones", fontsize=11)

    if i == 0:
        ax.set_ylabel("% of cells in cluster", fontsize=10)
    else:
        ax.set_ylabel("")

    ax.tick_params(axis='y', labelsize=9)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    ax.axvline(-0.5, color="black", linewidth=1)

# Save the faceted LOWESS figure
fig.savefig("../../05_plots/16_CD4analysis/12_CD4_PB_LOESS_trends_by_L3_cluster_facet_catpat.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()

# %%%%%%%%%%%%%%%%%%%%%
# --- Figure 8:  LOESS regression of CSF ---
# %%%%%%%%%%%%%%%%%%%%%
# Get unique groups
catpats = ["RRMS", "Control"]
n_facets = len(catpats)

# Prepare faceted subplots
fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)
if n_facets == 1:
    axes = [axes]  # Ensure axes is iterable

for i, cat in enumerate(catpats):
    ax = axes[i]
    
    # Subset the data
    df_cat = pivot_table_L3_csf_plot[pivot_table_L3_csf_plot["catpat"] == cat].reset_index(drop=True)
    x_vals = np.arange(len(df_cat))

    # LOWESS plot per cluster
    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        cluster_label = l3_cluster.replace("_pct", "")
        y_vals = df_cat[l3_cluster].values
        smoothed = lowess(y_vals, x_vals, frac=0.35)
        color = l3_color_dict_csf.get(cluster_label, "#333333")
        ax.plot(smoothed[:, 0], smoothed[:, 1], label=cluster_label, color=color, linewidth=2)

    # Style
    ax.set_xticks([])
    # ax.set_yticks([0, 20, 40, 60, 80])
    ax.set_ylim(0, 100)
    ax.tick_params(axis='y', labelsize=9)
    ax.set_title(f"{cat} clones", fontsize=11)
    
    if i == 0:
        ax.set_ylabel("% of cells in cluster", fontsize=10)
    else:
        ax.set_ylabel("")

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    ax.axvline(-0.5, color="black", linewidth=1)


# Save final plot
fig.savefig("../../05_plots/16_CD4analysis/14_CD4_CSF_LOESS_trends_by_L3_cluster_facet_catpat.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()



In [ ]:
## Save the compartment dataframe
CSF_and_PB_clone_pct = pivot_table_comp.copy()
print("Check length: ", len(CSF_and_PB_clone_pct))

# Plot 8: Annotating CSF and PBbiased CD4 T cell clones for DEG

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 1: Data Preparation and Filtering
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Make a data copy and filter that
sdata_highfrequent = sdata.copy()

# Combined criteria:
comb_sets = set5

# Make clone_handle categorical
sdata_highfrequent["gex"].obs["clone_handle"] = sdata_highfrequent["gex"].obs["clone_handle"].astype("category")

# Filter for combined clone set & remove certain donors
mu.pp.filter_obs(sdata_highfrequent["gex"], "clone_handle", lambda x: (x.isin(comb_sets)))
mu.pp.filter_obs(sdata_highfrequent["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 2: Compute Compartment Percentages
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Compute % contribution of each compartment per clone
sdata_highfrequent["gex"].obs["Compartment_pct"] = 100 * sdata_highfrequent["gex"].obs["type_x_clone_count"] / sdata_highfrequent["gex"].obs["total_clone_count"]

# Sanity check: number of unique clones
print("I have this many unique clones: ", len(sdata_highfrequent["gex"].obs["clone_handle"].drop_duplicates()))

# Create input dataframe for pivoting
df_input = sdata_highfrequent["gex"].obs[["clone_handle", "Compartment", "Compartment_pct"]]

# Pivot the data so compartments become columns
pivot_table_comp = df_input.pivot_table(index="clone_handle", columns="Compartment", values="Compartment_pct", fill_value=0)

# Rename columns
pivot_table_comp.columns = [f"{col}.pct" for col in pivot_table_comp.columns]

# Sort to show CSF-high clones first
pivot_table_comp = pivot_table_comp.sort_values(by="CSF.pct", ascending=False)  #Stores the compartment-wise dataset for plotting
clone_order = pivot_table_comp.index.tolist() #Will be used for L3_ann plots later
num_bars_comp = len(pivot_table_comp)

#Clone order PB: 
pivot_table_comp.sort_values(by="CSF.pct", ascending=False)
clone_order_pb = pivot_table_comp[pivot_table_comp["CSF.pct"] != 100].sort_values(by="CSF.pct", ascending=False).index.tolist()  #Stores the compartment-wise dataset for plotting
clone_order_csf = pivot_table_comp[pivot_table_comp["PB.pct"] != 100].sort_values(by="CSF.pct", ascending=False).index.tolist()  #Stores the compartment-wise dataset for plotting

# %%%%%%%%%%%%%%%%%%%
# Define highlight clone sets
# %%%%%%%%%%%%%%%%%%
CSF14_clones = clone_order[91:141]
PB14_clones = clone_order[-26:]  
highlight_clones = set(CSF14_clones + PB14_clones)
print(len(PB14_clones), " PB-biased clones, and this many ", len(CSF14_clones), "CSF-biased clones")
# highlight_clones = set(clone_biased_list)

# Export these clones: 
CSF14_clones_series = pd.Series(CSF14_clones)
CSF14_clones_series.to_csv(r"..\..\06_csv_outputs\Clones\2026_4_CD4_CSFbiased_clonehandles_ForCSF.csv", index=False)

PB14_clones_series = pd.Series(PB14_clones)
PB14_clones_series.to_csv(r"..\..\06_csv_outputs\Clones\2026_4_CD4_PBbiased_clonehandles_ForCSF.csv", index=False)


# %%%%%%%%%%%%%%%%%%%%%
# --- Figure 1+2: clones selected for differential gene expression
# %%%%%%%%%%%%%%%%%%%%%

# Set up subplots
fig1, ax1 = plt.subplots(1, 1, figsize=(8, 4), constrained_layout=True)
x = np.arange(len(pivot_table_comp))
width = 1

csf_colors = ["#4B8AC1" if clone in highlight_clones else "#E0E0E0" for clone in pivot_table_comp.index]
pb_colors = ["#C06A6A" if clone in highlight_clones else "#D8D8D8" for clone in pivot_table_comp.index]

ax1.bar(x, pivot_table_comp["CSF.pct"], width, label="CSF", color=csf_colors)  #Here I replace color="skyblue" with a list of colors according to the number of "top" clones were selected
ax1.bar(x, pivot_table_comp["PB.pct"], width, bottom=pivot_table_comp["CSF.pct"], label="PB", color=pb_colors) #Here I replace color="#C06A6A" with a list of colors according to the number of "top" clones were selected

### Kilian's ratio
#Add a horizontal line that represent the significance threshold for enrichment in CSF
ax1.axhline(y=83.33, color="black", ls="dashed", linewidth=0.5)  #This corresponds to 0.45 significance threshold calculated earlier

#Polish the plot
ax1.set_title(f"Clones w. more than 20 observed cells in total\n"
              f"CSF vs PB composition per clone\n{len(pivot_table_comp)} clones", fontsize=12)
ax1.set_xlabel("Clone Handle")
ax1.set_ylabel("Percentage")
ax1.set_xticks([])
for spine in ["top", "right", "left", "bottom"]:
    ax1.spines[spine].set_visible(False)
ax1.axvline(-0.5, color="black", linewidth=1)
ax1.legend(title="Compartment", bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)

# Export the figure
plt.tight_layout()
fig1.savefig("../../05_plots/16_CD4analysis/16_CD4_CSF_PB_stackbar_highlighted.pdf", bbox_inches='tight', format="pdf", dpi=400)
plt.show()

### Umap - Highlighting CSF-biased CD4 T cells in PB

In [ ]:
print(len(CSF14_clones))
print(len(PB14_clones))

In [ ]:
# Set figure params
sc.settings.set_figure_params(figsize=(4.3, 4))

# Define highlight clone sets
CSF14_clones 
PB14_clones

#For visualization
sdata["gex"].obs["top_vs_bot_clones2"] = pd.Categorical(
                      np.where(sdata["gex"].obs["clone_handle"].isin(PB14_clones), "PB-biased", 
                                np.where((sdata["gex"].obs["clone_handle"].isin(CSF14_clones)) & (sdata["gex"].obs["Compartment"] == "PB"), "CSF-biased in PB", 
                                         np.where(sdata["gex"].obs["clone_handle"].isin(CSF14_clones), "CSF-biased in CSF", "unselected"))))

#For later DEGs
sdata["gex"].obs["top_vs_bot_clones"] = pd.Categorical(
                      np.where(sdata["gex"].obs["clone_handle"].isin(PB14_clones), "PB-biased", 
                                np.where(sdata["gex"].obs["clone_handle"].isin(CSF14_clones), "CSF-biased",  "unselected")))


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# UMAP Plot
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Set figure parameters
sc.settings.set_figure_params() #Back to default

# Generate the UMAP figure and capture it
fig = sc.pl.umap(
    sdata["gex"],
    color=["top_vs_bot_clones2"],
    groups=["CSF-biased in PB", "CSF-biased in CSF", "PB-biased"],
    palette={"CSF-biased in CSF" : "#4B8AC1",  "CSF-biased in PB" : "#FD8D3C", "PB-biased" : "#C06A6A",  "unselected":"lightgrey"},
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=40,
    return_fig=True
)

# Save the figure
fig.savefig("../../05_plots/16_CD4analysis/17_showallcells_CD4_CSF_PB_UMAP_top_vs_bottom_clones.pdf", bbox_inches='tight', format="pdf", dpi = 400)

plt.show()



# Plot 9: Vulcano plot - DEGs
## CSF biased vs PB biased excluding rare transcripts

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Differential gene expression
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
sdata_pb = sdata.copy()
mu.pp.filter_obs(sdata_pb["gex"], "Compartment", lambda x: (x == "PB")) 

## Define the filtered to alter: 
sdata_pb_filtered = sdata_pb["gex"].copy()

# --- Filter genes before differential gene expression ---
## Step 1: Generate lists of expressed genes (>X%) for each of the 3 populations of observed cells T cells, B cells and interacting cells

## --- Expression in CSF-biased T cells
adata = sdata_pb["gex"][sdata_pb["gex"].obs["top_vs_bot_clones"] == "CSF-biased"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

## Report /lookup
print(fraction_df[fraction_df["names"] == "PDCD1"])

## --- Expression in PB-biased T cells
adata = sdata_pb["gex"][sdata_pb["gex"].obs["top_vs_bot_clones"] == "PB-biased"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

## Report /lookup
print(fraction_df[fraction_df["names"] == "PDCD1"])

#The total set of genes
gene_set_expressed = set(gene_list_10pct_1+gene_list_10pct_2)
print(len(gene_set_expressed))

# -----------------------------
# Step 2: exclude unwanted genes
# -----------------------------
genes = sdata_pb["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))
exclude_genes = set(genes[exclude_mask])
genes_to_keep = sorted(gene_set_expressed - exclude_genes)
print("Final genes:", len(genes_to_keep))

# -----------------------------
# Step 3: subset AnnData correctly
# -----------------------------
sdata_pb_filtered = sdata_pb_filtered[:, genes_to_keep].copy()

# Perform naive differential gene expression
sc.tl.rank_genes_groups(sdata_pb_filtered, groupby="top_vs_bot_clones", groups=["CSF-biased"], reference="PB-biased", key_added="CSF_biased_gene_set", method="wilcoxon")
sc.tl.rank_genes_groups(sdata_pb_filtered, groupby="top_vs_bot_clones", groups=["PB-biased"], reference="CSF-biased", key_added="PB_biased_gene_set", method="wilcoxon")

In [ ]:
# Display initial results
sc.pl.rank_genes_groups_dotplot(
     sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=30, key="CSF_biased_gene_set")
sc.pl.rank_genes_groups_dotplot(
     sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=30, key="PB_biased_gene_set")


In [ ]:
## Quick insoect
result = sc.get.rank_genes_groups_df(sdata_pb_filtered, group=None, key="CSF_biased_gene_set").copy()
result.sort_values("pvals_adj", ascending=True).head(20)

In [ ]:
# %%% 
# Vulcano plot function
# %%%

# # From: https://www.sc-best-practices.org/conditions/differential_gene_expression.html
LOG_FOLD_CHANGE = 0.5
PVALUE = -np.log10(0.05)  #Higher than p_adj 0.05. 
Add_labels = []
Add_labels = ['GZMK', 'CCR7', "GAPDH", "CCR5", 'CD27', "CD28", 'GPR183', 'CXCR4', "CXCR3", "CCR5", "MS4A1", "AC004448.2", "CLDND1", "CCL5", "KIR2DL4", "KIR2DL1", "PASK", "DKK3", "COTL1", "SARAF", "PPP1R14B", "NKG7", 
              'GZMB', 'ZNF683', 'GZMH', 'KLRD1',  'CX3CR1', 'FNDC3B',  'KLRC2', "TYROBP", "AOAH", "GBP5", "S100A4", "EFHD1", "HLA-C", "EFHD2", "MYL12A", "ST8SIA1", "ITGB1", "LGALS1", "GNLY", "CD52", "IL32", "IRF1", "HOPX", 
              "ZEB2", "FCRL6", "PRF1", "NKG7", "DUSP2", "IL7R", "EEF1A1", "SARAF", "ZFP36", "NELL2", "PDE4B", "PHACTR2" ]

VOLCANO_PALETTE = {
    "neg": "#D89A52",   # e.g. CSF-biased (negative logFC)
    "pos": "#C06A6A",   # e.g. PB-biased (positive logFC)
    "ns":  "lightgrey"
}

def volcano_plot(adata, group_key, title=None, palette=VOLCANO_PALETTE):
    result = sc.get.rank_genes_groups_df(
        adata, group=None, key=group_key
    ).copy()

    #Lookup
    print(result[result["names"] == "PDCD1"])

    result["-log_padj"] = -np.log10(result["pvals_adj"].astype(float))

    #Remove dots with a very low log2fc
    result=result[(result["logfoldchanges"] > 0.25) | (result["logfoldchanges"] < -0.25)]

    #Drop stuff that is na
    result = result.dropna(subset="names")

    # Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_padj"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_padj"] <= PVALUE)
    ]

    # Split significant genes by direction
    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.grid(True, zorder=0)

    # Non-significant genes
    sb.regplot(
        x=nonsig["logfoldchanges"],
        y=nonsig["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "alpha": 0.5, "zorder": 2},
        ax=ax,
        color=palette["ns"]
    )

    # Positively enriched
    sb.regplot(
        x=sig_pos["logfoldchanges"],
        y=sig_pos["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["pos"],
        label="Positive enrichment"
    )

    # Negatively enriched
    sb.regplot(
        x=sig_neg["logfoldchanges"],
        y=sig_neg["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["neg"],
        label="Negative enrichment"
    )

    # Rasterize only the dots
    # for coll in ax.collections:
    #     coll.set_rasterized(True)

    # Gene labels
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        ax.text(
            row["logfoldchanges"],
            row["-log_padj"],
            row["names"],
            fontsize=8,
            ha="right",
            va="bottom",
            zorder=4
        )

    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log padj")

    #Set limits
    ax.set_xlim(-8, 8)

    if title is None:
        title = group_key.replace("_", " ")
    ax.set_title(title)

    ax.legend(frameon=False)
    sb.despine()
    ax.grid(False)
    plt.tight_layout()

    plt.savefig(
        "../../05_plots/13_final_plots/17_CD4_vulcanoPlot_CSFvPB_bias.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=400
    )
    plt.show()

volcano_plot(sdata_pb_filtered, group_key="PB_biased_gene_set", title="<-- CSF-biased & PB-biased -->") 

## Broken axes

In [ ]:
def volcano_plot(adata, group_key, title=None, palette=VOLCANO_PALETTE):
    result = sc.get.rank_genes_groups_df(
        adata, group=None, key=group_key
    ).copy()

    #Lookup
    print(result[result["names"] == "PDCD1"])

    result["-log_padj"] = -np.log10(result["pvals_adj"].astype(float))

    #Remove dots with a very low log2fc
    result=result[(result["logfoldchanges"] > 0.25) | (result["logfoldchanges"] < -0.25)]

    #Drop stuff that is na
    result = result.dropna(subset="names")

    # Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_padj"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_padj"] <= PVALUE)
    ]

    # Split significant genes by direction
    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    # --- Broken axis setup ---
    fig = plt.figure(figsize=(8, 6))
    bax = brokenaxes(
        xlims=((-8, 8), (27, 32)),  # <-- adjust to your ranges
        hspace=0.05
    )

    # --- Plotting ---
    bax.scatter(
        nonsig["logfoldchanges"],
        nonsig["-log_padj"],
        s=60, alpha=0.5, color=palette["ns"], zorder=2
    )

    bax.scatter(
        sig_pos["logfoldchanges"],
        sig_pos["-log_padj"],
        s=60, color=palette["pos"], label="Positive enrichment", zorder=3
    )

    bax.scatter(
        sig_neg["logfoldchanges"],
        sig_neg["-log_padj"],
        s=60, color=palette["neg"], label="Negative enrichment", zorder=3
    )

    # --- Gene labels ---
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        bax.text(
            row["logfoldchanges"],
            row["-log_padj"],
            row["names"],
            fontsize=8,
            ha="right",
            va="bottom",
            zorder=4
        )

    # --- Labels ---
    bax.set_xlabel("log2 FC")
    bax.set_ylabel("-log padj")

    # --- Finish up ---
    bax.grid(False)

    plt.savefig(
        "../../05_plots/16_CD4analysis/19_CD4_vulcanoPlot_CSFvPB_before_custom_filters_broken_axes.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=400
    )
    plt.show()


volcano_plot(
    sdata_pb_filtered,
    group_key="PB_biased_gene_set",
    title="<-- CSF-biased & PB-biased -->"
)

## Export CSF/PB-biased DEGs

In [ ]:
datPBbiased = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None, key ="PB_biased_gene_set").dropna(subset="names")
datPBbiased.to_csv("..\\..\\06_csv_outputs\\DEG\\2026_08_CD4_RAW_CSFnPB_biased_DEGs.csv")

datCSFbiased = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None, key ="CSF_biased_gene_set").dropna(subset="names")
datCSFbiased.to_csv("..\\..\\06_csv_outputs\\DEG\\2026_08_CD4_RAW_CSFnPB_biased_DEGs.csv")


## Score calculation

In [ ]:
### --- Get gene lists of CSF/PB-biased gene scores ---
CSFbiased_genescore = datCSFbiased[(datCSFbiased["logfoldchanges"]> 1) & (datCSFbiased["pvals_adj"]<0.01)]
PBbiased_genescore = datPBbiased[(datPBbiased["logfoldchanges"]> 1) & (datPBbiased["pvals_adj"]<0.01)]

# --- Export gene scores --- 
CSFbiased_genescore.to_csv("../../06_csv_outputs/DEG/CD4_CSF_biased_filtered_DEGs.csv")
PBbiased_genescore.to_csv("../../06_csv_outputs/DEG/CD4_PB_biased_filtered_DEGs.csv")

print(len(CSFbiased_genescore), " genes were CSF-biased")
print(len(PBbiased_genescore), " genes were CSF-biased")

# --- Calculate scores --- 
sc.tl.score_genes(sdata["gex"], 
                      CSFbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='CSF permissive score')

sc.tl.score_genes(sdata["gex"], 
                      PBbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='PB resident score')
# --- Display scores --- 
p1 = sc.pl.umap(
    sdata["gex"], 
    color=["CSF permissive score", "PB resident score"], 
    size=10, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
p1.savefig("../../05_plots/13_final_plots/35_CD4_CSF_PB_biased_score_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

## Correlation

In [ ]:
# Step 1: Extract relevant data
df = sdata["gex"].obs[["donor","clone_handle", "total_clone_count", "CSF permissive score", "PB resident score", "Compartment"]].copy()
df = df[df["Compartment"] == "PB"]
df = df[~df["donor"].isin(["Pt4", "Pt5"])]
df = df[df["total_clone_count"] > 5]
df["clone_handle"]= df["clone_handle"].cat.remove_unused_categories()

# Step 2: Compute average score per clone
clone_avg = df.groupby("clone_handle")[["PB resident score", "CSF permissive score"]].mean().reset_index()

# Step 3: Reorder based on clone_order
# clone_reordered = clone_avg.loc[clone_order].reset_index(drop=True)

# Step 4: Drop NaNs in CSF permissive score
# clone_reordered = clone_reordered.dropna(subset=["CSF permissive score"])

## Add information on CSF percentage
Merge_pct = pivot_table_comp.reset_index()
clone_avg = clone_avg.merge(Merge_pct, how="left", on="clone_handle")

## Filter on clones with some CSF-pct
# clone_avg = clone_avg[clone_avg["CSF.pct"] > 0]

# Prepare data
x_vals = clone_avg["CSF.pct"].values/100          # Before: np.arange(len(clone_reordered))
y_vals = clone_avg["CSF permissive score"].values   #clone_reordered["CSF permissive score"].values

# Compute Spearman correlation
rho, pval = spearmanr(x_vals, y_vals)

# Step 5: Plot using seaborn for regression with confidence interval
fig, ax = plt.subplots(figsize=(4.5, 4))

# Scatterplot + linear regression with CI
sb.regplot(x=x_vals, y=y_vals, ax=ax, ci=95, robust=True, scatter_kws={"color": "grey", "s": 40}, line_kws={"color": "black", "linestyle": "--"})

# Horizontal dashed line at y = 0
ax.axhline(y=0, linestyle='--', linewidth=0.5, color='lightgrey')

# Add Spearman correlation text
ax.text(0.01, 0.95,
        f"Spearman ρ = {rho:.2f}\np = {pval:.2e}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8))

# Formatting
ax.set_title("CSF permissive score vs clone rank\nwith Linear Regression + 95% CI and Spearman Correlation")
ax.set_xlabel("Fraction of cells in CSF/clone")
ax.set_ylabel("Average CSF-biased score per clone")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# ax.set_ylim(0.75, 2)
ax.grid(False)

# Save the figure
fig.savefig("../../05_plots/13_final_plots/22_CD4_clone_score_regression_CI_spearman.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()

In [ ]:
# Step 1: Extract relevant data
df = sdata["gex"].obs[["donor","clone_handle", "total_clone_count", "CSF permissive score", "PB resident score", "Compartment"]].copy()
df = df[df["Compartment"] == "PB"]
df = df[~df["donor"].isin(["Pt4", "Pt5"])]
df = df[df["total_clone_count"] > 5]
df["clone_handle"]= df["clone_handle"].cat.remove_unused_categories()

# Step 2: Compute average score per clone
clone_avg = df.groupby("clone_handle")[["PB resident score", "CSF permissive score"]].mean().reset_index()

# Step 3: Reorder based on clone_order
# clone_reordered = clone_avg.loc[clone_order].reset_index(drop=True)

# Step 4: Drop NaNs in CSF permissive score
# clone_reordered = clone_reordered.dropna(subset=["CSF permissive score"])

## Add information on CSF percentage
Merge_pct = pivot_table_comp.reset_index()
clone_avg = clone_avg.merge(Merge_pct, how="left", on="clone_handle")

## Filter on clones with some CSF-pct
# clone_avg = clone_avg[clone_avg["PB.pct"] > 0]

# Prepare data
x_vals = clone_avg["PB.pct"].values/100          # Before: np.arange(len(clone_reordered))
y_vals = clone_avg["PB resident score"].values   #clone_reordered["CSF permissive score"].values

# Compute Spearman correlation
rho, pval = spearmanr(x_vals, y_vals)

# Step 5: Plot using seaborn for regression with confidence interval
fig, ax = plt.subplots(figsize=(4.5, 4))

# Scatterplot + linear regression with CI
sb.regplot(x=x_vals, y=y_vals, ax=ax, ci=95, robust=True, scatter_kws={"color": "grey", "s": 40}, line_kws={"color": "black", "linestyle": "--"})

# Horizontal dashed line at y = 0
ax.axhline(y=0, linestyle='--', linewidth=0.5, color='lightgrey')

# Add Spearman correlation text
ax.text(0.01, 0.95,
        f"Spearman ρ = {rho:.2f}\np = {pval:.2e}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8))

# Formatting
ax.set_xlabel("Fraction of cells in PB")
ax.set_ylabel("Average PB-biased score")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# ax.set_ylim(0.75, 2)
ax.grid(False)

# Save the figure
fig.savefig("../../05_plots/13_final_plots/22_clone_score_regression_CI_spearman_PB.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()

# Plot 10: The overlap between CD4 and CD8 CSF signatures

In [ ]:
### Initial plot with the filtered DEGs ###

#CD8 signature
CD8T_CSFbiased = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_filtered_DEGs.csv")
CD8T_CSFbiased["source"] = "CD8 T cells"
print("CD8 CSF-biased score has this many members: ", len(CD8T_CSFbiased))

CD4T_CSFbiased = pd.read_csv("../../06_csv_outputs/DEG/CD4_CSF_biased_filtered_DEGs.csv")
CD4T_CSFbiased["source"] = "CD4 T cells"
print("CD4 CSF-biased score has this many members: ", len(CD4T_CSFbiased))

## Merge
combined_df = pd.concat([CD8T_CSFbiased, CD4T_CSFbiased])[["names", "scores", "logfoldchanges", "pvals", "pvals_adj", "source"]]

## Pivot multiple colums
wide_df = combined_df.pivot(
    index="names",
    columns="source",
    values=["logfoldchanges", "pvals_adj"]
)

wide_df.columns = [
    f"{metric}_{source}" for metric, source in wide_df.columns
]

wide_df = wide_df.reset_index()

## Replace all NAs with 0
wide_df["pvals_adj_CD4 T cells"] = wide_df["pvals_adj_CD4 T cells"].fillna(1)
wide_df["pvals_adj_CD8 T cells"] = wide_df["pvals_adj_CD8 T cells"].fillna(1)
wide_df = wide_df.fillna(0)
wide_df.sort_values("logfoldchanges_CD8 T cells", ascending=False).head(10)

#Define colors
wide_df["catcolor"] = np.where((wide_df["pvals_adj_CD4 T cells"] < 0.01) & (wide_df["pvals_adj_CD8 T cells"] < 0.05), "p < 0.01 in both CD4 and CD8", 
                               np.where((wide_df["pvals_adj_CD4 T cells"] < 0.01), "p < 0.01 in CD4", 
                                        np.where((wide_df["pvals_adj_CD8 T cells"] < 0.01), "p < 0.01 in CD8", "ns"))) 

color_palette = {"p < 0.01 in both CD4 and CD8" : "red", 
                 "p < 0.01 in CD8" : "orange",  
                "p < 0.01 in CD4" : "blue", 
}

##  Plot
df = wide_df.copy()

df = df[df["catcolor"]!= "ns"]

fig, ax = plt.subplots(figsize=(9, 6))

sb.scatterplot(
    data=df,
    x="logfoldchanges_CD4 T cells",
    y="logfoldchanges_CD8 T cells",
    hue="catcolor",
    palette=color_palette,
    ax=ax
)

# Example: annotate specific genes
# e.g. strongest CD8 signals
topCD8 = df.nlargest(15, "logfoldchanges_CD8 T cells")["names"].tolist()
topCD4 = df.nlargest(15, "logfoldchanges_CD4 T cells")["names"].tolist()
genes_to_label = set(topCD8 + topCD4 + ["CXCR3", "CXCR4"]) 

for _, row in df[df["names"].isin(genes_to_label)].iterrows():
    ax.text(
        row["logfoldchanges_CD4 T cells"],
        row["logfoldchanges_CD8 T cells"],
        row["names"],
        fontsize=9,
        ha="left",
        va="bottom"
    )


plt.axhline(0, linestyle="--", lw=0.5)
plt.axvline(0, linestyle="--", lw=0.5)

plt.xlabel("CD4 logFC")
plt.ylabel("CD8 logFC")

plt.title("CD4 vs CD8 CSF-biased genes")
## move the legend outside of the plotting area. 
plt.legend(
    bbox_to_anchor=(1.05, 1),  # position outside right
    loc="upper left",
    borderaxespad=0
)

sb.despine()
plt.grid(False)

plt.tight_layout()
plt.show()


In [ ]:
#CD8 signature
CD8T_CSFbiased = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_raw_DEGs.csv")
CD8T_CSFbiased["source"] = "CD8 T cells"
print("CD8 CSF-biased score has this many members: ", len(CD8T_CSFbiased))

CD4T_CSFbiased = pd.read_csv("../../06_csv_outputs/DEG/CD4_CSF_biased_raw_DEGs.csv")
CD4T_CSFbiased["source"] = "CD4 T cells"
print("CD4 CSF-biased score has this many members: ", len(CD4T_CSFbiased))

## Merge
combined_df = pd.concat([CD8T_CSFbiased, CD4T_CSFbiased])[["names", "scores", "logfoldchanges", "pvals", "pvals_adj", "source"]]

## Pivot multiple colums
wide_df = combined_df.pivot(
    index="names",
    columns="source",
    values=["logfoldchanges", "pvals_adj"]
)

wide_df.columns = [
    f"{metric}_{source}" for metric, source in wide_df.columns
]

wide_df = wide_df.reset_index()

## Replace all NAs with 0
wide_df["pvals_adj_CD4 T cells"] = wide_df["pvals_adj_CD4 T cells"].fillna(1)
wide_df["pvals_adj_CD8 T cells"] = wide_df["pvals_adj_CD8 T cells"].fillna(1)
wide_df = wide_df.fillna(0)
wide_df.sort_values("logfoldchanges_CD8 T cells", ascending=False).head(10)

#Define colors
wide_df["catcolor"] = np.where((wide_df["pvals_adj_CD4 T cells"] < 0.05) & (wide_df["pvals_adj_CD8 T cells"] < 0.05), "padj < 0.05 in both CD4 and CD8", 
                               np.where((wide_df["pvals_adj_CD4 T cells"] < 0.05), "padj < 0.05 in CD4", 
                                        np.where((wide_df["pvals_adj_CD8 T cells"] < 0.05), "padj < 0.05 in CD8", "ns"))) 

color_palette = {"padj < 0.05 in both CD4 and CD8" : "red", 
                 "padj < 0.05 in CD8" : "orange",  
                "padj < 0.05 in CD4" : "blue", 
                 "ns" : "darkgrey", 
}

In [ ]:
from matplotlib.ticker import MultipleLocator

# --- Data ---
df = wide_df.copy()
df = df[df["catcolor"] != "ns"]

# --- Broken axis setup ---
fig = plt.figure(figsize=(7, 6))
bax = brokenaxes(
    width_ratios=[1, 7],
    xlims=((-32, -20), (-7, 10)),
    hspace=0.05
)

# --- Scatter ---
for key, subdf in df.groupby("catcolor"):
    bax.scatter(
        subdf["logfoldchanges_CD4 T cells"],
        subdf["logfoldchanges_CD8 T cells"],
        label=key,
        color=color_palette[key],
        s=20
    )

# Example: annotate specific genes
# e.g. strongest CD8 signals
topCD8 = df.nlargest(10, "logfoldchanges_CD8 T cells")["names"].tolist()
topCD4 = df.nlargest(10, "logfoldchanges_CD4 T cells")["names"].tolist()
botCD8 = df.nsmallest(10, "logfoldchanges_CD8 T cells")["names"].tolist()
botCD4 = df.nsmallest(10, "logfoldchanges_CD4 T cells")["names"].tolist()
genes_to_label = set(topCD8 + topCD4 + botCD8 + botCD4 + ["CXCR3", "CXCR4"]) 

for _, row in df[df["names"].isin(genes_to_label)].iterrows():
    bax.text(
        row["logfoldchanges_CD4 T cells"],
        row["logfoldchanges_CD8 T cells"],
        row["names"],
        fontsize=7,
        ha="left",
        va="bottom"
    )


# --- Reference lines ---
bax.axhline(0, linestyle="--", lw=0.5)
bax.axvline(0, linestyle="--", lw=0.5)

# --- Labels ---
bax.set_xlabel("CD4 logFC")
bax.set_ylabel("CD8 logFC")
bax.set_title("CD4 vs CD8 CSF-biased genes")

# --- Legend ---
bax.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

# --- 🔑 FORCE ticks ---
for ax in bax.axs:
    ax.xaxis.set_major_locator(MultipleLocator(2))   # major ticks every 2
    ax.xaxis.set_minor_locator(MultipleLocator(1))   # minor ticks every 1
    ax.tick_params(axis='x', which='major', length=6)
    ax.tick_params(axis='x', which='minor', length=3)

# --- Clean ---
bax.grid(False)

# plt.tight_layout()
plt.savefig("../../05_plots/16_CD4analysis/24_CD4_CD8_overlapping_DEGs_CSFbias_UpDown.pdf", format="pdf", dpi=400)
plt.show()

# Plot 11: B-LCL reactive clones

## Delta fractions

In [ ]:
## Load B-LCL reactive TCRb chains
clone_raw = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")

## All reactive clone handles here have a larger fold change than 1

In [ ]:
## Make a list of B_LCL reactive TCRb chains
# clone_list_BLCL1 = clone_raw["aaSeqCDR3"].drop_duplicates().tolist()
# clone_list_BLCL2 = clone_raw[clone_raw["FC0_19"] > 10]["aaSeqCDR3"].drop_duplicates().tolist()
# clone_list_BLCL3 = clone_raw[clone_raw["FC0_19"] > 100]["aaSeqCDR3"].drop_duplicates().tolist()

## Annotate sdata["gex"].obs
# sdata["gex"].obs["LCL_reactivity1"] = pd.Categorical(np.where(sdata["gex"].obs["IR_VDJ_1_junction_aa"].isin(clone_list_BLCL1), "B-LCL reactive", "Non-reactive"))
# sdata["gex"].obs["LCL_reactivity2"] = pd.Categorical(np.where(sdata["gex"].obs["IR_VDJ_1_junction_aa"].isin(clone_list_BLCL2), "B-LCL reactive", "Non-reactive"))
# sdata["gex"].obs["LCL_reactivity3"] = pd.Categorical(np.where(sdata["gex"].obs["IR_VDJ_1_junction_aa"].isin(clone_list_BLCL3), "B-LCL reactive", "Non-reactive"))

## Make a list of B_LCL reactive TCRb chains
clone_list_BLCL1 = clone_raw["Clone_handle"].drop_duplicates().tolist()
clone_list_BLCL2 = clone_raw[clone_raw["FC0_19"] > 10]["Clone_handle"].drop_duplicates().tolist()
clone_list_BLCL3 = clone_raw[clone_raw["FC0_19"] > 100]["Clone_handle"].drop_duplicates().tolist()

#define a clone_handle_beta (donor + TCRb_1)
sdata["gex"].obs["clone_handle_beta"] = sdata["gex"].obs["donor"].astype(str) + "_" + sdata["gex"].obs["IR_VDJ_1_junction_aa"].astype(str)

## Annotate sdata["gex"].obs
sdata["gex"].obs["LCL_reactivity1"] = pd.Categorical(np.where(sdata["gex"].obs["clone_handle_beta"].isin(clone_list_BLCL1), "B-LCL reactive", "Non-reactive"))
sdata["gex"].obs["LCL_reactivity2"] = pd.Categorical(np.where(sdata["gex"].obs["clone_handle_beta"].isin(clone_list_BLCL2), "B-LCL reactive", "Non-reactive"))
sdata["gex"].obs["LCL_reactivity3"] = pd.Categorical(np.where(sdata["gex"].obs["clone_handle_beta"].isin(clone_list_BLCL3), "B-LCL reactive", "Non-reactive"))
sdata.update()

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig_umap = sc.pl.umap(sdata["gex"], color=["LCL_reactivity1", "LCL_reactivity2", "LCL_reactivity3"], groups=["B-LCL reactive"], size = 40, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, cmap="RdBu_r", return_fig=True)
fig_umap.savefig( "../../05_plots/16_CD4analysis/26_umap_B_LCL_reactive.pdf", bbox_inches="tight", format="pdf", dpi=400)
plt.show()

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig_umap = sc.pl.umap(sdata["gex"], color=["donor"], groups=["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"], size = 10, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, cmap="RdBu_r", return_fig=True)
fig_umap.savefig( "../../05_plots/15_CD4annotation/18_umap_B_LCL_reactive_donor.pdf", bbox_inches="tight", format="pdf", dpi=400)
plt.show()

In [ ]:
## groups
# df = sdata["gex"].obs[(sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])) & (sdata["gex"].obs["Compartment"] == "PB")][["LCL_reactivity1", "L3_ann", "leiden_1.8"]].value_counts().reset_index()
# df = df[~df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"])]
df = sdata["gex"].obs[sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])][["LCL_reactivity1", "L3_ann", "leiden_1.8"]].value_counts().reset_index()
df["L3_ann"] = df["L3_ann"].cat.remove_unused_categories()
df["leiden_1.8"] = df["leiden_1.8"].cat.remove_unused_categories()

## Percentages
df["freq"] = df["count"] / df.groupby("LCL_reactivity1")["count"].transform("sum")

# Set desired order
order = ["Non-reactive", "B-LCL reactive"]

df["LCL_reactivity1"] = pd.Categorical(
    df["LCL_reactivity1"],
    categories=order,
    ordered=True
)

# Sort so pivot respects order
df = df.sort_values("LCL_reactivity1")

# Pivot to remove the background percentage
df_pivot = df.pivot(index="leiden_1.8", columns="LCL_reactivity1", values="freq")
df_pivot["delta"] = df_pivot["B-LCL reactive"] - df_pivot["Non-reactive"]
df_pivot = df_pivot.sort_values("delta").reset_index()

# --- plot ---
plt.figure(figsize=(8, 3.5))

sb.barplot(
    data=df_pivot,
    x="leiden_1.8",
    y="delta",
    color="lightgrey"   # base color (we'll override below if needed)
)
                                                                
# Add horizontal reference line at 0
plt.axhline(0, color="black", linestyle="--", linewidth=1)

plt.xticks()
plt.ylabel("Δ Fraction\n(B-LCL reactive - Non-reactive)")
plt.xlabel("")
plt.ylim(-0.2, 0.2)

sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig( "../../05_plots/16_CD4analysis/28_NormalizedColumnPlot.pdf", bbox_inches="tight", format="pdf", dpi=400)
plt.show()

# # Pivot for stacked barplot
# df_pivot = df.pivot(index="LCL_reactivity1", columns="L3_ann", values="freq")

# #Pass colors to the df
# colors = [color_dict2[col] for col in df_pivot.columns]

# df_pivot.plot(kind="bar", stacked=True, figsize=(8.25,6), color=colors)

# plt.ylabel("Frequency")
# plt.xlabel("")
# plt.title("Compartment = PB")
# plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="L3_ann")
# sb.despine()
# plt.grid(False)
# plt.tight_layout()
# plt.savefig( "../../05_plots/16_CD4analysis/27_StackedBarPB.pdf", bbox_inches="tight", format="pdf", dpi=400)
# plt.show()

## Proportion among CSF, PB-biased clones

In [ ]:
# Lists containing names of CSF and PB-biased clones
CSF14_clones 
PB14_clones

## Get obs and annotate
df = sdata["gex"].obs[sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])]
df["biased_ann"] = np.where(df["clone_handle"].isin(CSF14_clones), "CSF-biased", 
                            np.where(df["clone_handle"].isin(PB14_clones), "PB-biased", "Neutral"))

## Count
df = df[["LCL_reactivity1", "biased_ann"]].value_counts().reset_index()

## Percentages
df["freq"] = df["count"] / df.groupby("LCL_reactivity1")["count"].transform("sum")


In [ ]:
# Set desired order
order = ["Non-reactive", "B-LCL reactive"]

df["LCL_reactivity1"] = pd.Categorical(
    df["LCL_reactivity1"],
    categories=order,
    ordered=True
)

# Sort so pivot respects order
df = df.sort_values("LCL_reactivity1")

# Pivot to remove the background percentage
df_pivot = df.pivot(index="biased_ann", columns="LCL_reactivity1", values="freq")
df_pivot["delta"] = df_pivot["B-LCL reactive"] - df_pivot["Non-reactive"]
df_pivot = df_pivot.sort_values("delta").reset_index()

# --- plot ---
plt.figure(figsize=(8, 3.5))

sb.barplot(
    data=df_pivot,
    x="biased_ann",
    y="delta",
    color="lightgrey"   # base color (we'll override below if needed)
)
                                                                
# Add horizontal reference line at 0
plt.axhline(0, color="black", linestyle="--", linewidth=1)

plt.xticks()
plt.ylabel("Δ Fraction\n(B-LCL reactive - Non-reactive)")
plt.xlabel("")

sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig( "../../05_plots/16_CD4analysis/28_NormalizedColumnPlot_biased_ann.pdf", bbox_inches="tight", format="pdf", dpi=400)
plt.show()

df

## Visualize LCL-reactive in CSF and PB as bar graphs

In [ ]:
## Get obs and annotate
df = sdata["gex"].obs[sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])]

## Focus on hyperexpanded (more than 5 cells)
df = df[df["total_clone_count"] > 5]

## Report
print("There are this many expanded clonotypes to analyse:", len(df[["clone_handle", "LCL_reactivity1"]].drop_duplicates()))

## select columns and tally
count_df = df[["clone_handle", "LCL_reactivity1", "Compartment", "catpat"]].value_counts().reset_index()

## Pivot wider
wide_df = count_df.pivot_table(index=["clone_handle", "LCL_reactivity1", "catpat"], columns="Compartment", values="count", fill_value=0).reset_index()
wide_df.sort_values("LCL_reactivity1")

## calculate precentages
wide_df["CSF.pct"] = 100*wide_df["CSF"] / (wide_df["CSF"] + wide_df["PB"]) 
wide_df["PB.pct"] = 100*wide_df["PB"] / (wide_df["CSF"] + wide_df["PB"]) 

# Drop unannotated
plot_df = wide_df.dropna(subset="LCL_reactivity1").copy()

# Sort by CSF contribution
plot_df = plot_df.sort_values("CSF.pct", ascending=False)

# Select relevant columns
plot_long = plot_df.melt(
    id_vars=["clone_handle", "LCL_reactivity1", "catpat"],
    value_vars=["CSF.pct", "PB.pct"],
    var_name="Compartment",
    value_name="Percent"
)

# Pivot for stacked barplot
plot_pivot = (
    plot_long
    .pivot(
        index="clone_handle",
        columns="Compartment",
        values="Percent"
    )
    .fillna(0)
)

plot_pivot = plot_pivot[["CSF.pct", "PB.pct"]]

# Preserve CSF sorting
clone_order = plot_df["clone_handle"].drop_duplicates()
plot_pivot = plot_pivot.reindex(clone_order)

# Colors
virus_colors = {
    "B-LCL reactive": "#1F77B4",
    "Non-reactive": "#D3D3D3",
}

catpat_colors = {
    "NIC": "#FFFFFF",
    "MS": "#000000"
}


# Match annotations to clone order
annotation_df = (
    plot_df
    .drop_duplicates("clone_handle")
    .set_index("clone_handle")
    .loc[clone_order]
)


virus_track = annotation_df["LCL_reactivity1"]
catpat_track = annotation_df["catpat"]

virus_rgb = [
    plt.matplotlib.colors.to_rgb(virus_colors[v])
    for v in virus_track
]

catpat_rgb = [
    plt.matplotlib.colors.to_rgb(catpat_colors[v])
    for v in catpat_track
]


# Figure with two annotation tracks
fig, (ax, ax_virus, ax_catpat) = plt.subplots(
    nrows=3,
    figsize=(6, 6),
    gridspec_kw={
        "height_ratios": [8, 0.4, 0.4]
    },
    sharex=True
)


# --------------------
# Main stacked barplot
# --------------------
plot_pivot.plot(
    kind="bar",
    stacked=True,
    color=["#4B8AC1", "#C06A6A"],
    width=1,
    edgecolor="none",
    ax=ax
)

ax.set_ylabel("Compartment (%)")
ax.set_xlabel("")
ax.set_ylim(0, 100)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    handles=[
        Patch(facecolor="#4B8AC1", label="CSF"),
        Patch(facecolor="#C06A6A", label="PB"),
    ],
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

ax.grid(False)


# --------------------
# Virus annotation strip
# --------------------
ax_virus.imshow(
    np.array([virus_rgb]),
    aspect="auto",
    interpolation="nearest"
)

ax_virus.set_yticks([])
ax_virus.set_xticks([])


# --------------------
# Cohort annotation strip
# --------------------
ax_catpat.imshow(
    np.array([catpat_rgb]),
    aspect="auto",
    interpolation="nearest"
)

ax_catpat.set_yticks([])
ax_catpat.set_xticks([])


# Remove annotation axes borders
for annotation_ax in [ax_virus, ax_catpat]:
    for spine in annotation_ax.spines.values():
        spine.set_visible(False)


# Legends
virus_legend = [
    Patch(facecolor=color, label=virus)
    for virus, color in virus_colors.items()
    if virus in virus_track.values
]

ax_virus.legend(
    handles=virus_legend,
    title="Virus specificity",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

catpat_legend = [
    Patch(facecolor=color, label=group)
    for group, color in catpat_colors.items()
    if group in catpat_track.values
]

ax_catpat.legend(
    handles=catpat_legend,
    title="Cohort",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)


plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig( "../../05_plots/16_CD4analysis/29_AnnotationOfHyperexpandedClones.pdf", bbox_inches="tight", format="pdf", dpi=400)
plt.show()

# Plot 12: PB-biased and CSF-biassed score per cluster

## CSF-biased score

In [ ]:
# --- Export gene scores --- 
CSFbiased_genescore = pd.read_csv("../../06_csv_outputs/DEG/CD4_CSF_biased_filtered_DEGs.csv")
PBbiased_genescore = pd.read_csv("../../06_csv_outputs/DEG/CD4_PB_biased_filtered_DEGs.csv")

print(len(CSFbiased_genescore), " genes were CSF-biased")
print(len(PBbiased_genescore), " genes were CSF-biased")

# --- Calculate scores --- 
sc.tl.score_genes(sdata["gex"], 
                      CSFbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='CSF permissive score')

sc.tl.score_genes(sdata["gex"], 
                      PBbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='PB resident score')

In [ ]:
## Kilian: Split by PB and CSF
##
order_csf = ["7", "2", "11", "12", "14", "19", "1"]

# ## get the right compartment 
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "Compartment", lambda x: x == "CSF")
mu.pp.filter_obs(sdata_temp["gex"], "leiden_1.8", lambda x: x.isin(order_csf)) ## 

# # Remove all unused categories
for col in sdata_temp["gex"].obs.select_dtypes(include="category").columns:
    sdata_temp["gex"].obs[col] = sdata_temp["gex"].obs[col].cat.remove_unused_categories()

# Extract obs and sort groups by median
obs = sdata_temp["gex"].obs.copy()
medians = obs.groupby("leiden_1.8")["CSF permissive score"].median()
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata_temp["gex"],
        keys="CSF permissive score",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("CSF-biased score")
plt.ylim(-0.5, 1.8)

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_leiden_vs_CSFscore_csf.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
## Kilian: Split by PB and CSF
##
order_csf = ["7", "2", "11", "12", "14", "19", "1"]

# ## get the right compartment 
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "Compartment", lambda x: x == "PB")
mu.pp.filter_obs(sdata_temp["gex"], "leiden_1.8", lambda x: ~x.isin(order_csf)) ## 

# # Remove all unused categories
for col in sdata_temp["gex"].obs.select_dtypes(include="category").columns:
    sdata_temp["gex"].obs[col] = sdata_temp["gex"].obs[col].cat.remove_unused_categories()

# Extract obs and sort groups by median
obs = sdata_temp["gex"].obs.copy()
medians = obs.groupby("leiden_1.8")["CSF permissive score"].median()
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata_temp["gex"],
        keys="CSF permissive score",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("CSF-biased score")
plt.ylim(-0.5, 1.8)

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_leiden_vs_CSFscore_pb.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

## PB-biased score

In [ ]:
## Kilian: Split by PB and CSF
##
order_csf = ["7", "2", "11", "12", "14", "19", "1"]

# ## get the right compartment 
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "Compartment", lambda x: x == "CSF")
mu.pp.filter_obs(sdata_temp["gex"], "leiden_1.8", lambda x: x.isin(order_csf)) ## 

# # Remove all unused categories
for col in sdata_temp["gex"].obs.select_dtypes(include="category").columns:
    sdata_temp["gex"].obs[col] = sdata_temp["gex"].obs[col].cat.remove_unused_categories()

# Extract obs and sort groups by median
obs = sdata_temp["gex"].obs.copy()
medians = obs.groupby("leiden_1.8")["PB resident score"].median()
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata_temp["gex"],
        keys="PB resident score",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("PB-biased score")
plt.ylim(-0.5, 1.8)

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_leiden_vs_PBscore_csf.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
## Kilian: Split by PB and CSF
##
order_csf = ["7", "2", "11", "12", "14", "19", "1"]

# ## get the right compartment 
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "Compartment", lambda x: x == "PB")
mu.pp.filter_obs(sdata_temp["gex"], "leiden_1.8", lambda x: ~x.isin(order_csf)) ## 

# # Remove all unused categories
for col in sdata_temp["gex"].obs.select_dtypes(include="category").columns:
    sdata_temp["gex"].obs[col] = sdata_temp["gex"].obs[col].cat.remove_unused_categories()

# Extract obs and sort groups by median
obs = sdata_temp["gex"].obs.copy()
medians = obs.groupby("leiden_1.8")["PB resident score"].median()
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata_temp["gex"],
        keys="PB resident score",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("PB-biased score")
plt.ylim(-0.5, 1.8)

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_leiden_vs_PBscore_pb.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

# Plot 13: % LCL-reactive in PB and CSF, respectively

In [ ]:
## Get obs and annotate
df = sdata["gex"].obs[sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])]

# Remove all unused categories
for col in df.select_dtypes(include="category").columns:
    df[col] = df[col].cat.remove_unused_categories()


## Count
df = df[["LCL_reactivity1", "Compartment", "catpat", "donor"]].value_counts().reset_index()

## Percentages
df["freq"] = df["count"] / df.groupby("LCL_reactivity1")["count"].transform("sum")

# Make sure ordering is consistent
df["catpat"] = pd.Categorical(
    df["catpat"],
    categories=["NIC", "MS"],
    ordered=True
)

df["Compartment"] = pd.Categorical(
    df["Compartment"],
    categories=["PB", "CSF"],
    ordered=True
)

## We're only interested in B-LCL reactive
df = df[df["LCL_reactivity1"] == "B-LCL reactive"]

plt.figure(figsize=(5, 5))

sb.boxplot(
    data=df,
    x="Compartment",
    y="freq",
    hue="catpat",
    order=["PB", "CSF"],
    hue_order=["NIC", "MS"],
    width=0.65,
    fliersize=0,
    boxprops={"facecolor": "white", "edgecolor": "black"},
    whiskerprops={"color": "black"},
    capprops={"color": "black"},
    medianprops={"color": "black"},
)

sb.stripplot(
    data=df,
    x="Compartment",
    y="freq",
    hue="catpat",
    order=["PB", "CSF"],
    hue_order=["NIC", "MS"],
    dodge=True,
    jitter=0.15,
    size=6,
    edgecolor="black",
    linewidth=0.5,
    alpha=0.8,
)

# Remove duplicate legends
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title=None, frameon=False)

plt.xlabel("")
plt.ylabel("Freq. of B-LCL reactive cells")
sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig("../../05_plots/16_CD4analysis/XXX_LCL_reactive_MS_NIC_CSF_PB.pdf")
plt.show()

# Plot 12: T helper signatures per cluster

## Tr1 score

In [ ]:
sdata["gex"].obs.columns

In [ ]:
# ## get the right compartment 

# Extract obs and sort groups by median
obs = sdata["gex"].obs
# medians = obs.groupby("leiden_1.8")["Th1 (Ostkampf et al. 2022)"].median()
# sorted_groups = medians.sort_values().index.tolist()
sorted_groups = ["0", "4", "5", "6", "8", "10", "3", "9", "17", "15", "18", "13", "16", "20", "1", "2", "7", "11", "12", "14", "19"]

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="Tr1+ cells (De Simone et al 2021)",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("Tr1 score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_leiden_vs_Tr1.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
# ## get the right compartment 

# Extract obs and sort groups by median
obs = sdata["gex"].obs
# medians = obs.groupby("leiden_1.8")["Th1 (Ostkampf et al. 2022)"].median()
# sorted_groups = medians.sort_values().index.tolist()
# sorted_groups = ["0", "4", "5", "6", "8", "10", "3", "9", "17", "15", "18", "13", "16", "20", "1", "2", "7", "11", "12", "14", "19"]

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (3, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="Tr1+ cells (De Simone et al 2021)",
        groupby="LCL_reactivity1",
        # order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("Th1 score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_LCL_vs_Tr1.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
from scipy.stats import ttest_ind

## Extract relevant cells and their scores
obs = sdata["gex"].obs.copy()
obs[["LCL_reactivity1", "Tr1+ cells (De Simone et al 2021)"]]

# Split into pos and negative
tr1_pos = obs.loc[
    obs["LCL_reactivity1"] == "B-LCL reactive",
    "Tr1+ cells (De Simone et al 2021)"
]

tr1_neg = obs.loc[
    obs["LCL_reactivity1"] == "Non-reactive",
    "Tr1+ cells (De Simone et al 2021)"
]

# Independent two-sample t-test
stat, pval = ttest_ind(
    tr1_pos,
    tr1_neg,
    equal_var=False,  # Welch's t-test
    alternative="two-sided"
)

print(f"Tr1+ n = {len(tr1_pos)}")
print(f"Tr1- n = {len(tr1_neg)}")
print(f"Welch's t = {stat:.2f}")
print(f"p = {pval:.3e}")

## Th1 score per cluster

In [ ]:
# ## get the right compartment 

# Extract obs and sort groups by median
obs = sdata["gex"].obs
# medians = obs.groupby("leiden_1.8")["Th1 (Ostkampf et al. 2022)"].median()
# sorted_groups = medians.sort_values().index.tolist()
sorted_groups = ["0", "4", "5", "6", "8", "10", "3", "9", "17", "15", "18", "13", "16", "20", "1", "2", "7", "11", "12", "14", "19"]

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="Th1 (Ostkampf et al. 2022)",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("Th1 score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_leiden_vs_Th1.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
# ## get the right compartment 

# Extract obs and sort groups by median
obs = sdata["gex"].obs
# medians = obs.groupby("leiden_1.8")["Th1 (Ostkampf et al. 2022)"].median()
# sorted_groups = medians.sort_values().index.tolist()
# sorted_groups = ["0", "4", "5", "6", "8", "10", "3", "9", "17", "15", "18", "13", "16", "20", "1", "2", "7", "11", "12", "14", "19"]

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (3, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="Th1 (Ostkampf et al. 2022)",
        groupby="LCL_reactivity1",
        # order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("Th1 score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_LCL_vs_Th1.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
from scipy.stats import ttest_ind

## Extract relevant cells and their scores
obs = sdata["gex"].obs.copy()
obs[["LCL_reactivity1", "Th1 (Ostkampf et al. 2022)"]]

# Split into pos and negative
tr1_pos = obs.loc[
    obs["LCL_reactivity1"] == "B-LCL reactive",
    "Th1 (Ostkampf et al. 2022)"
]

tr1_neg = obs.loc[
    obs["LCL_reactivity1"] == "Non-reactive",
    "Th1 (Ostkampf et al. 2022)"
]

# Independent two-sample t-test
stat, pval = ttest_ind(
    tr1_pos,
    tr1_neg,
    equal_var=False,  # Welch's t-test
    alternative="two-sided"
)

print(f"T+ n = {len(tr1_pos)}")
print(f"T- n = {len(tr1_neg)}")
print(f"Welch's t = {stat:.2f}")
print(f"p = {pval:.3e}")

## Th2 score per cluster

In [ ]:
# ## get the right compartment 

# Extract obs and sort groups by median
obs = sdata["gex"].obs
# medians = obs.groupby("leiden_1.8")["Th1 (Ostkampf et al. 2022)"].median()
# sorted_groups = medians.sort_values().index.tolist()
sorted_groups = ["0", "4", "5", "6", "8", "10", "3", "9", "17", "15", "18", "13", "16", "20", "1", "2", "7", "11", "12", "14", "19"]

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="Th2 (Ostkampf et al. 2022)",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("Th2 score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_leiden_vs_Th2.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

## Tfh score per cluster

In [ ]:
# ## get the right compartment 

# Extract obs and sort groups by median
obs = sdata["gex"].obs
# medians = obs.groupby("leiden_1.8")["Th1 (Ostkampf et al. 2022)"].median()
# sorted_groups = medians.sort_values().index.tolist()
sorted_groups = ["0", "4", "5", "6", "8", "10", "3", "9", "17", "15", "18", "13", "16", "20", "1", "2", "7", "11", "12", "14", "19"]

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="Tfh (Ostkampf et al. 2022)",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("Tfh score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_leiden_vs_Tfh.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
# ## get the right compartment 

# Extract obs and sort groups by median
obs = sdata["gex"].obs
# medians = obs.groupby("leiden_1.8")["Th1 (Ostkampf et al. 2022)"].median()
# sorted_groups = medians.sort_values().index.tolist()
# sorted_groups = ["0", "4", "5", "6", "8", "10", "3", "9", "17", "15", "18", "13", "16", "20", "1", "2", "7", "11", "12", "14", "19"]

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (3, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="Tfh (Ostkampf et al. 2022)",
        groupby="LCL_reactivity1",
        # order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("Thfh score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_LCL_vs_Tfh.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
from scipy.stats import ttest_ind

## Extract relevant cells and their scores
obs = sdata["gex"].obs.copy()
obs[["LCL_reactivity1", "Tfh (Ostkampf et al. 2022)"]]

# Split into pos and negative
tr1_pos = obs.loc[
    obs["LCL_reactivity1"] == "B-LCL reactive",
    "Tfh (Ostkampf et al. 2022)"
]

tr1_neg = obs.loc[
    obs["LCL_reactivity1"] == "Non-reactive",
    "Tfh (Ostkampf et al. 2022)"
]

# Independent two-sample t-test
stat, pval = ttest_ind(
    tr1_pos,
    tr1_neg,
    equal_var=False,  # Welch's t-test
    alternative="two-sided"
)

print(f"T+ n = {len(tr1_pos)}")
print(f"T- n = {len(tr1_neg)}")
print(f"Welch's t = {stat:.2f}")
print(f"p = {pval:.3e}")

## Th17 score per cluster

In [ ]:
# ## get the right compartment 

# Extract obs and sort groups by median
obs = sdata["gex"].obs
# medians = obs.groupby("leiden_1.8")["Th1 (Ostkampf et al. 2022)"].median()
# sorted_groups = medians.sort_values().index.tolist()
sorted_groups = ["0", "4", "5", "6", "8", "10", "3", "9", "17", "15", "18", "13", "16", "20", "1", "2", "7", "11", "12", "14", "19"]

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (6, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="Th17 (Ostkampf et al. 2022)",
        groupby="leiden_1.8",
        order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("Th17 score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_leiden_vs_Th17.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
# ## get the right compartment 

# Extract obs and sort groups by median
obs = sdata["gex"].obs
# medians = obs.groupby("leiden_1.8")["Th1 (Ostkampf et al. 2022)"].median()
# sorted_groups = medians.sort_values().index.tolist()
# sorted_groups = ["0", "4", "5", "6", "8", "10", "3", "9", "17", "15", "18", "13", "16", "20", "1", "2", "7", "11", "12", "14", "19"]

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (2, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="Th17 (Ostkampf et al. 2022)",
        groupby="LCL_reactivity1",
        # order=sorted_groups,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
# plt.xticks(rotation=0, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=0, linewidth=0.5, color="black", linestyle="--")
plt.ylabel("Th17 score")

# Rasterize violin and box shapes for PDF
# for col in ax.collections:
#     col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_CD4_LCL_vs_Th17.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
from scipy.stats import ttest_ind

## Extract relevant cells and their scores
obs = sdata["gex"].obs.copy()
obs[["LCL_reactivity1", "Th17 (Ostkampf et al. 2022)"]]

# Split into pos and negative
tr1_pos = obs.loc[
    obs["LCL_reactivity1"] == "B-LCL reactive",
    "Th17 (Ostkampf et al. 2022)"
]

tr1_neg = obs.loc[
    obs["LCL_reactivity1"] == "Non-reactive",
    "Th17 (Ostkampf et al. 2022)"
]

# Independent two-sample t-test
stat, pval = ttest_ind(
    tr1_pos,
    tr1_neg,
    equal_var=False,  # Welch's t-test
    alternative="two-sided"
)

print(f"T+ n = {len(tr1_pos)}")
print(f"T- n = {len(tr1_neg)}")
print(f"Welch's t = {stat:.2f}")
print(f"p = {pval:.3e}")

# OLD CODE


# Differentially expressed genes PB-biased vs CSF-biased
Calculate raw differentially expressed genes

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Differential gene expression
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
sdata_pb = sdata.copy()

mu.pp.filter_obs(sdata_pb["gex"], "Compartment", lambda x: (x == "PB")) 

# Perform differential gene expression
sc.tl.rank_genes_groups(sdata_pb["gex"], groupby="top_vs_bot_clones", groups=["CSF-biased"], reference="PB-biased", n_genes=50, method="wilcoxon", key_added="CSF_biased_gene_set")
sc.tl.rank_genes_groups(sdata_pb["gex"], groupby="top_vs_bot_clones",  groups=["PB-biased"], reference="CSF-biased", n_genes=50, method="wilcoxon", key_added="PB_biased_gene_set")

In [ ]:
# WHich cells are being compared: 
sc.pl.umap(
    sdata_pb["gex"],
    color=["leiden_1.7"],
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=40
)

In [ ]:
# Display initial results
sc.pl.rank_genes_groups_dotplot(
      sdata_pb["gex"], groupby="top_vs_bot_clones", standard_scale="var", n_genes=30, key="CSF_biased_gene_set")
sc.pl.rank_genes_groups_dotplot(
      sdata_pb["gex"], groupby="top_vs_bot_clones", standard_scale="var", n_genes=30, key="PB_biased_gene_set")


# Remove TCR, Ribosomal genes, and sex-specific genes before calculating DEGs

In [ ]:
# Remove TCR-related and ribosomal genes
# TCR genes often start with 'TRAV', 'TRBV', 'TRGV', etc.
# Ribosomal genes often start with 'RPS' or 'RPL'

sdata_pb_filtered = sdata_pb.copy()

# IMPORTANT TO SUBSET ONLY ON CELLS OF INTEREST- because the filter_rank_genes behaves odd if there are cells outside the groups (target and reference) that express a gene of interest. CCR7 kept getting filtered out because naive cells/other cells were part of the analysis
# mu.pp.filter_obs(sdata_pb_filtered["gex"], "top_vs_bot_clones", lambda x: (x != "unselected"))   #Going back to using memory cells as background for filtering

# Create a mask for genes to exclude
genes = sdata_pb["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_pb_filtered = sdata_pb_filtered["gex"][:, genes_to_keep]

# Run rank_genes_groups with filtered genes
sc.tl.rank_genes_groups(
    sdata_pb_filtered,
    groupby="top_vs_bot_clones",
    groups=["CSF-biased"],
    reference="PB-biased",
    method="wilcoxon", 
    key_added="CSF_biased_gene_set"
)

sc.tl.rank_genes_groups(
    sdata_pb_filtered,
    groupby="top_vs_bot_clones",
    groups=["PB-biased"],
    reference="CSF-biased",
    method="wilcoxon", 
    key_added="PB_biased_gene_set"
)

sc.pl.rank_genes_groups_dotplot(
      sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=80, key="CSF_biased_gene_set")
sc.pl.rank_genes_groups_dotplot(
      sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=80, key="PB_biased_gene_set")

# %%%% 
# Export all differentially expressed genes
# %%%%

Export_DEG = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="CSF_biased_gene_set")
Export_DEG.to_csv("..\\..\\06_csv_outputs\\DEG\\2026_4_CD4_CSF_DEG_without_ribosomal_or_mitoch_genes.csv")

In [ ]:
# ### --- Derive a list of genes that are expressed by more than 5% of cells under consideration (CSF+PB biased) ---
# sc.tl.rank_genes_groups(
#     sdata_pb_filtered,
#     groupby="top_vs_bot_clones",
#     groups=["unselected"],
#     reference="rest",
#     method="wilcoxon", 
#     pts=True,
#     key_added="GS_unsel"
# )

In [ ]:
# ## Getting the fraction of cells that express a given gene in CSF-biased + PB-biased uses the "pts_rest" fraction here. 
# ### pts_rest is ""Only if `reference` is set to `'rest'`. Fraction of cells from the union of the rest of each group, expressing the genes.
# res = sdata_pb_filtered.uns["GS_unsel"]

# df1 = pd.DataFrame({
#     "rest_fra" : res["pts_rest"]["unselected"],
# })

# ## (A) gene_list_highly_expressed
# gene_list_highly_expressed = df1[df1["rest_fra"] > 0.05].index.tolist()

In [ ]:
# %%% 
# Vulcano plot function
# %%%

# # From: https://www.sc-best-practices.org/conditions/differential_gene_expression.html
LOG_FOLD_CHANGE = 0.0
PVALUE = -np.log10(0.05)  #Higher than p_adj 0.05. 
Add_labels = []
Add_labels = ['GZMK', 'CCR7', "GAPDH", 'CD27','GPR183', 'CXCR4', "CXCR3", "CCR5", "MS4A1", "AC004448.2", "CLDND1", 
              'GZMB', 'ZNF683', 'GZMH', 'CX3CR1', 'FNDC3B', "TYROBP", "AOAH", "GBP5", "STAT1", "IRF1", 
              "ZEB2", "FCRL6", "PRF1", "EEF1A1", "NKG7", "PPP1R14B", "EEF1B2", "LTB", "B2M"
             ]

VOLCANO_PALETTE = {
    "neg": "#8A7DBF",   # e.g. CSF-biased (negative logFC)
    "pos": "#C06A6A",   # e.g. PB-biased (positive logFC)
    "ns":  "lightgrey"
}

def volcano_plot(adata, group_key, title=None, palette=VOLCANO_PALETTE):
    result = sc.get.rank_genes_groups_df(
        adata, group=None, key=group_key
    ).copy()

    result["-log_padj"] = -np.log10(result["pvals_adj"].astype(float))

    #Remove genes with a very low log2fc
    # result=result[(result["logfoldchanges"] > 1) | (result["logfoldchanges"] < -1)]

    #Remove outlier genes with a -log_padj = 0. These are unecessary to visualize, and stem from difficult to detect gene expression
    # result=result[result["-log_padj"] != 0]    
    
    #Remove genes that are not in the list of commonly detected genes
    # result=result[result["names"].isin(gene_list_highly_expressed)]

    #Drop stuff that is na
    result = result.dropna(subset="names")

    # Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_padj"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_padj"] <= PVALUE)
    ]

    # Split significant genes by direction
    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.grid(True, zorder=0)

    # Non-significant genes
    sb.regplot(
        x=nonsig["logfoldchanges"],
        y=nonsig["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "alpha": 0.5, "zorder": 2},
        ax=ax,
        color=palette["ns"]
    )

    # Positively enriched
    sb.regplot(
        x=sig_pos["logfoldchanges"],
        y=sig_pos["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["pos"],
        label="Positive enrichment"
    )

    # Negatively enriched
    sb.regplot(
        x=sig_neg["logfoldchanges"],
        y=sig_neg["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["neg"],
        label="Negative enrichment"
    )

    # Rasterize only the dots
    # for coll in ax.collections:
    #     coll.set_rasterized(True)

    # Gene labels
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        ax.text(
            row["logfoldchanges"],
            row["-log_padj"],
            row["names"],
            fontsize=8,
            ha="right",
            va="bottom",
            zorder=4
        )

    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log padj")

    if title is None:
        title = group_key.replace("_", " ")
    ax.set_title(title)

    ax.legend(frameon=False)
    sb.despine()
    ax.grid(False)
    plt.tight_layout()

    plt.savefig(
        "../../05_plots/16_CD4analysis/19_CD4_vulcanoPlot_CSFvPB_before_custom_filters.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=400
    )
    plt.show()

volcano_plot(sdata_pb_filtered, group_key="PB_biased_gene_set", title="<-- CSF-biased & PB-biased -->") 

# Broken axes

In [ ]:
def volcano_plot(adata, group_key, title=None, palette=VOLCANO_PALETTE):
    result = sc.get.rank_genes_groups_df(
        adata, group=None, key=group_key
    ).copy()

    result["-log_padj"] = -np.log10(result["pvals_adj"].astype(float))

    # Filters
    # result = result[(result["logfoldchanges"] > 1) | (result["logfoldchanges"] < -1)]
    result=result[result["-log_padj"] != 0]    
    result = result.dropna(subset="names")

    # Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_padj"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_padj"] <= PVALUE)
    ]

    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    # --- Broken axis setup ---
    fig = plt.figure(figsize=(8, 6))
    bax = brokenaxes(
        xlims=((-8, 7), (27, 32)),  # <-- adjust to your ranges
        hspace=0.05
    )

    # --- Plotting ---
    bax.scatter(
        nonsig["logfoldchanges"],
        nonsig["-log_padj"],
        s=60, alpha=0.5, color=palette["ns"], zorder=2
    )

    bax.scatter(
        sig_pos["logfoldchanges"],
        sig_pos["-log_padj"],
        s=60, color=palette["pos"], label="Positive enrichment", zorder=3
    )

    bax.scatter(
        sig_neg["logfoldchanges"],
        sig_neg["-log_padj"],
        s=60, color=palette["neg"], label="Negative enrichment", zorder=3
    )

    # --- Gene labels ---
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        bax.text(
            row["logfoldchanges"],
            row["-log_padj"],
            row["names"],
            fontsize=8,
            ha="right",
            va="bottom",
            zorder=4
        )

    # --- Labels ---
    bax.set_xlabel("log2 FC")
    bax.set_ylabel("-log padj")

    # --- Finish up ---
    bax.grid(False)

    plt.savefig(
        "../../05_plots/16_CD4analysis/19_CD4_vulcanoPlot_CSFvPB_before_custom_filters_broken_axes.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=400
    )
    plt.show()


volcano_plot(
    sdata_pb_filtered,
    group_key="PB_biased_gene_set",
    title="<-- CSF-biased & PB-biased -->"
)

In [ ]:
df = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="PB_biased_gene_set")
df.sort_values("pvals_adj")

# Export raw CSF-biased DEGs

In [ ]:
## Get two dataframes of positive lfc values associated with CSF-biased and PB-biased T cells. Concatenate them and plot
datCSF_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="CSF_biased_gene_set").dropna(subset="names")
datPB_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="PB_biased_gene_set").dropna(subset="names")

#EXPORT unfiltered gene scores
datCSF_raw.to_csv("../../06_csv_outputs/DEG/CD4_CSF_biased_raw_DEGs.csv")
datPB_raw.to_csv("../../06_csv_outputs/DEG/CD4_PB_biased_raw_DEGs.csv")

# Filter and score calculation

In [ ]:
#Old filtering method:
## --- The custom filtering approach is inspired by sc.tl.filter_rank_genes_groups()
#      This scanpy function kept removing genes with higher expression outside the target group "rest". What I want is to only remove genes that are higher expressed among "unselected". 

sc.tl.filter_rank_genes_groups(
    sdata_pb_filtered,
    min_in_group_fraction=0.05, 
    max_out_group_fraction=1,   #Max is 1 - no filtering will take place
    min_fold_change=0,
    compare_abs=False,       #Compare_abs=True retains negative log2fc genes.. We specifically want upregulated genes for the next analysis
    key="CSF_biased_gene_set",
    key_added="CSF_biased_gene_set (w. filtering)",
)

sc.tl.filter_rank_genes_groups(
   sdata_pb_filtered,
    min_in_group_fraction=0.05, 
    max_out_group_fraction=1,   #fraction positive !! outside !! the population of interest be less than 0.9
    min_fold_change=0,      #To retain all information
    compare_abs=False,       #Compare_abs=True retains negative log2fc genes..  We specifically want upregulated genes for the next analysis.
    key="PB_biased_gene_set",
    key_added="PB_biased_gene_set (w. filtering)",
)

### --- Get gene lists of CSF/PB-biased gene scores ---
datCSF = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="CSF_biased_gene_set (w. filtering)").dropna(subset="names")
datPB = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="PB_biased_gene_set (w. filtering)").dropna(subset="names")

CSFbiased_genescore = datCSF[(datCSF["logfoldchanges"]> 1) & (datCSF["pvals_adj"]<0.05)]
PBbiased_genescore = datPB[(datPB["logfoldchanges"]> 1) & (datPB["pvals_adj"]<0.05)]

# --- Export gene scores --- 
CSFbiased_genescore.to_csv("../../06_csv_outputs/DEG/CD4_CSF_biased_filtered_DEGs.csv")
PBbiased_genescore.to_csv("../../06_csv_outputs/DEG/CD4_PB_biased_filtered_DEGs.csv")

# --- Calculate scores --- 
sc.tl.score_genes(sdata["gex"], 
                      CSFbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='CSF permissive score')

sc.tl.score_genes(sdata["gex"], 
                      PBbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='PB resident score')
# --- Display scores --- 
p1 = sc.pl.umap(
    sdata["gex"], 
    color=["CSF permissive score", "PB resident score"], 
    size=50, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
plt.show()
p1.savefig("../../05_plots/16_CD4analysis/20_umap_biased_scores.pdf", bbox_inches='tight', format="pdf", dpi = 400)

# Display CSF/PB-biased gene set

In [ ]:
print(f"{len(CSFbiased_genescore)} genes make up my CSF permissive score")
print(f"{len(PBbiased_genescore)} genes make up my PB resident score")

In [ ]:
sc.pl.dotplot(sdata_pb_filtered, CSFbiased_genescore["names"], groupby='top_vs_bot_clones', standard_scale="var", dendrogram=True, cmap="RdBu_r")

In [ ]:
sc.pl.dotplot(sdata_pb_filtered, PBbiased_genescore["names"], groupby='top_vs_bot_clones', dendrogram=True,standard_scale="var", cmap="RdBu_r")

# Correlation

In [ ]:
# Step 1: Extract relevant data
df = sdata["gex"].obs[["donor","clone_handle", "CSF permissive score", "PB resident score", "Compartment"]].copy()
df = df[df["Compartment"] == "PB"]
df = df[~df["donor"].isin(["Pt4", "Pt5"])]
df = df[df["clone_handle"].isin(set5)]
df["clone_handle"]= df["clone_handle"].cat.remove_unused_categories()

# Step 2: Compute average score per clone
clone_avg = df.groupby("clone_handle")[["PB resident score", "CSF permissive score"]].mean().reset_index()

# Step 3: Reorder based on clone_order
# clone_reordered = clone_avg.loc[clone_order].reset_index(drop=True)

# Step 4: Drop NaNs in CSF permissive score
# clone_reordered = clone_reordered.dropna(subset=["CSF permissive score"])

## Add information on CSF percentage
Merge_pct = CSF_and_PB_clone_pct.reset_index()
clone_avg = clone_avg.merge(Merge_pct, how="left", on="clone_handle")

## Filter on clones with some CSF-pct
# clone_avg = clone_avg[clone_avg["CSF.pct"] > 0]

# Prepare data
x_vals = clone_avg["CSF.pct"].values /100         # Before: np.arange(len(clone_reordered))
y_vals = clone_avg["CSF permissive score"].values   #clone_reordered["CSF permissive score"].values

# Compute Spearman correlation
rho, pval = spearmanr(x_vals, y_vals)

# Step 5: Plot using seaborn for regression with confidence interval
fig, ax = plt.subplots(figsize=(4.5, 4))

# Scatterplot + linear regression with CI
sb.regplot(x=x_vals, y=y_vals, ax=ax, ci=95, robust=True, scatter_kws={"color": "grey", "s": 20}, line_kws={"color": "black", "linestyle": "--"})

# Horizontal dashed line at y = 0
ax.axhline(y=0, linestyle='--', linewidth=0.5, color='lightgrey')

# Add Spearman correlation text
ax.text(0.01, 0.95,
        f"Spearman ρ = {rho:.2f}\np = {pval:.2e}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8))

# Formatting
ax.set_title("CSF permissive score vs clone rank\nwith Linear Regression + 95% CI and Spearman Correlation")
ax.set_xlabel("Fraction of cells in CSF/clone")
ax.set_ylabel("Average CSF permissive score")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# ax.axvline(-0.5, color="black", linewidth=1)
# ax.set_ylim(0.75, 2)
ax.grid(False)

# Save the figure
fig.savefig("../../05_plots/16_CD4analysis/21_CD4T_spearman_correlation_CSFbiased_CSFfraction.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()

# Outlier cells

In [ ]:
## 2 outlier clones
clone_avg[(clone_avg['CSF.pct'] > 60) & (clone_avg['CSF permissive score'] < 0.25) ]

In [ ]:
# Define clones to highlight
clones_of_interest = ["Pt18-29353", "Pt20-112690", "Pt12-17278", "Pt8-3830", "Pt1-69"]

# Define a new columns
sdata["gex"].obs["clone_highlight"] = pd.Categorical(np.where(sdata["gex"].obs["clone_handle"].isin(clones_of_interest), sdata["gex"].obs["clone_handle"], "unselected"))

# Palette
palette_highlight = {
    "unselected": "lightgrey",
    "Pt18-29353": "red",
    "Pt20-112690": "blue", 
    "Pt12-17278" : "orange", 
    "Pt8-3830" : "yellow", 
    "Pt1-69" : "black"
}

# Plot
sc.pl.umap(
    sdata["gex"],
    color="clone_highlight",
    groups=["Pt18-29353", "Pt20-112690", "Pt12-17278", "Pt8-3830", "Pt1-69"],
    palette=palette_highlight,
    size=50
)

In [ ]:
sdata["gex"].obs[sdata["gex"].obs["clone_handle"].isin(clones_of_interest)][["donor", "catpat"]].value_counts()

# The overlap between CD4 and CD8 CSF signatures

In [ ]:
### Initial plot with the filtered DEGs ###

#CD8 signature
CD8T_CSFbiased = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_filtered_DEGs.csv")
CD8T_CSFbiased["source"] = "CD8 T cells"
print("CD8 CSF-biased score has this many members: ", len(CD8T_CSFbiased))

CD4T_CSFbiased = pd.read_csv("../../06_csv_outputs/DEG/CD4_CSF_biased_filtered_DEGs.csv")
CD4T_CSFbiased["source"] = "CD4 T cells"
print("CD4 CSF-biased score has this many members: ", len(CD4T_CSFbiased))

## Merge
combined_df = pd.concat([CD8T_CSFbiased, CD4T_CSFbiased])[["names", "scores", "logfoldchanges", "pvals", "pvals_adj", "source"]]

## Pivot multiple colums
wide_df = combined_df.pivot(
    index="names",
    columns="source",
    values=["logfoldchanges", "pvals_adj"]
)

wide_df.columns = [
    f"{metric}_{source}" for metric, source in wide_df.columns
]

wide_df = wide_df.reset_index()

## Replace all NAs with 0
wide_df["pvals_adj_CD4 T cells"] = wide_df["pvals_adj_CD4 T cells"].fillna(1)
wide_df["pvals_adj_CD8 T cells"] = wide_df["pvals_adj_CD8 T cells"].fillna(1)
wide_df = wide_df.fillna(0)
wide_df.sort_values("logfoldchanges_CD8 T cells", ascending=False).head(10)

#Define colors
wide_df["catcolor"] = np.where((wide_df["pvals_adj_CD4 T cells"] < 0.01) & (wide_df["pvals_adj_CD8 T cells"] < 0.05), "p < 0.01 in both CD4 and CD8", 
                               np.where((wide_df["pvals_adj_CD4 T cells"] < 0.01), "p < 0.01 in CD4", 
                                        np.where((wide_df["pvals_adj_CD8 T cells"] < 0.01), "p < 0.01 in CD8", "ns"))) 

color_palette = {"p < 0.01 in both CD4 and CD8" : "red", 
                 "p < 0.01 in CD8" : "orange",  
                "p < 0.01 in CD4" : "blue", 
}

##  Plot
df = wide_df.copy()

df = df[df["catcolor"]!= "ns"]

fig, ax = plt.subplots(figsize=(9, 6))

sb.scatterplot(
    data=df,
    x="logfoldchanges_CD4 T cells",
    y="logfoldchanges_CD8 T cells",
    hue="catcolor",
    palette=color_palette,
    ax=ax
)

# Example: annotate specific genes
# e.g. strongest CD8 signals
topCD8 = df.nlargest(15, "logfoldchanges_CD8 T cells")["names"].tolist()
topCD4 = df.nlargest(15, "logfoldchanges_CD4 T cells")["names"].tolist()
genes_to_label = set(topCD8 + topCD4 + ["CXCR3", "CXCR4"]) 

for _, row in df[df["names"].isin(genes_to_label)].iterrows():
    ax.text(
        row["logfoldchanges_CD4 T cells"],
        row["logfoldchanges_CD8 T cells"],
        row["names"],
        fontsize=9,
        ha="left",
        va="bottom"
    )


plt.axhline(0, linestyle="--", lw=0.5)
plt.axvline(0, linestyle="--", lw=0.5)

plt.xlabel("CD4 logFC")
plt.ylabel("CD8 logFC")

plt.title("CD4 vs CD8 CSF-biased genes")
## move the legend outside of the plotting area. 
plt.legend(
    bbox_to_anchor=(1.05, 1),  # position outside right
    loc="upper left",
    borderaxespad=0
)

sb.despine()
plt.grid(False)

plt.tight_layout()
plt.show()


In [ ]:
#CD8 signature
CD8T_CSFbiased = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_raw_DEGs.csv")
CD8T_CSFbiased["source"] = "CD8 T cells"
print("CD8 CSF-biased score has this many members: ", len(CD8T_CSFbiased))

CD4T_CSFbiased = pd.read_csv("../../06_csv_outputs/DEG/CD4_CSF_biased_raw_DEGs.csv")
CD4T_CSFbiased["source"] = "CD4 T cells"
print("CD4 CSF-biased score has this many members: ", len(CD4T_CSFbiased))

## Merge
combined_df = pd.concat([CD8T_CSFbiased, CD4T_CSFbiased])[["names", "scores", "logfoldchanges", "pvals", "pvals_adj", "source"]]

## Pivot multiple colums
wide_df = combined_df.pivot(
    index="names",
    columns="source",
    values=["logfoldchanges", "pvals_adj"]
)

wide_df.columns = [
    f"{metric}_{source}" for metric, source in wide_df.columns
]

wide_df = wide_df.reset_index()

## Replace all NAs with 0
wide_df["pvals_adj_CD4 T cells"] = wide_df["pvals_adj_CD4 T cells"].fillna(1)
wide_df["pvals_adj_CD8 T cells"] = wide_df["pvals_adj_CD8 T cells"].fillna(1)
wide_df = wide_df.fillna(0)
wide_df.sort_values("logfoldchanges_CD8 T cells", ascending=False).head(10)

#Define colors
wide_df["catcolor"] = np.where((wide_df["pvals_adj_CD4 T cells"] < 0.05) & (wide_df["pvals_adj_CD8 T cells"] < 0.05), "padj < 0.05 in both CD4 and CD8", 
                               np.where((wide_df["pvals_adj_CD4 T cells"] < 0.05), "padj < 0.05 in CD4", 
                                        np.where((wide_df["pvals_adj_CD8 T cells"] < 0.05), "padj < 0.05 in CD8", "ns"))) 

color_palette = {"padj < 0.05 in both CD4 and CD8" : "red", 
                 "padj < 0.05 in CD8" : "orange",  
                "padj < 0.05 in CD4" : "blue", 
                 "ns" : "darkgrey", 
}

In [ ]:
from matplotlib.ticker import MultipleLocator

# --- Data ---
df = wide_df.copy()
df = df[df["catcolor"] != "ns"]

# --- Broken axis setup ---
fig = plt.figure(figsize=(7, 6))
bax = brokenaxes(
    width_ratios=[1, 7],
    xlims=((-32, -20), (-7, 10)),
    hspace=0.05
)

# --- Scatter ---
for key, subdf in df.groupby("catcolor"):
    bax.scatter(
        subdf["logfoldchanges_CD4 T cells"],
        subdf["logfoldchanges_CD8 T cells"],
        label=key,
        color=color_palette[key],
        s=20
    )

# Example: annotate specific genes
# e.g. strongest CD8 signals
topCD8 = df.nlargest(10, "logfoldchanges_CD8 T cells")["names"].tolist()
topCD4 = df.nlargest(10, "logfoldchanges_CD4 T cells")["names"].tolist()
botCD8 = df.nsmallest(10, "logfoldchanges_CD8 T cells")["names"].tolist()
botCD4 = df.nsmallest(10, "logfoldchanges_CD4 T cells")["names"].tolist()
genes_to_label = set(topCD8 + topCD4 + botCD8 + botCD4 + ["CXCR3", "CXCR4"]) 

for _, row in df[df["names"].isin(genes_to_label)].iterrows():
    bax.text(
        row["logfoldchanges_CD4 T cells"],
        row["logfoldchanges_CD8 T cells"],
        row["names"],
        fontsize=7,
        ha="left",
        va="bottom"
    )


# --- Reference lines ---
bax.axhline(0, linestyle="--", lw=0.5)
bax.axvline(0, linestyle="--", lw=0.5)

# --- Labels ---
bax.set_xlabel("CD4 logFC")
bax.set_ylabel("CD8 logFC")
bax.set_title("CD4 vs CD8 CSF-biased genes")

# --- Legend ---
bax.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

# --- 🔑 FORCE ticks ---
for ax in bax.axs:
    ax.xaxis.set_major_locator(MultipleLocator(2))   # major ticks every 2
    ax.xaxis.set_minor_locator(MultipleLocator(1))   # minor ticks every 1
    ax.tick_params(axis='x', which='major', length=6)
    ax.tick_params(axis='x', which='minor', length=3)

# --- Clean ---
bax.grid(False)

# plt.tight_layout()
plt.savefig("../../05_plots/16_CD4analysis/24_CD4_CD8_overlapping_DEGs_CSFbias_UpDown.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
##  Plot
df = wide_df.copy()

df = df[df["catcolor"]!= "ns"]
df = df[df["logfoldchanges_CD4 T cells"]>=0]
df = df[df["logfoldchanges_CD8 T cells"]>=0]
df = df[df["names"]!="POU3F1"]

fig, ax = plt.subplots(figsize=(9, 6))

sb.scatterplot(
    data=df,
    s=30,
    x="logfoldchanges_CD4 T cells",
    y="logfoldchanges_CD8 T cells",
    hue="catcolor",
    palette=color_palette,
    ax=ax
)

# Example: annotate specific genes
# e.g. strongest CD8 signals
topCD8 = df.nlargest(10, "logfoldchanges_CD8 T cells")["names"].tolist()
topCD4 = df.nlargest(10, "logfoldchanges_CD4 T cells")["names"].tolist()
genes_to_label = set(topCD8 + topCD4 + ["CXCR3", "CXCR4"]) 

for _, row in df[df["names"].isin(genes_to_label)].iterrows():
    ax.text(
        row["logfoldchanges_CD4 T cells"],
        row["logfoldchanges_CD8 T cells"],
        row["names"],
        fontsize=9,
        ha="left",
        va="bottom"
    )

ax.axhline(0, linestyle="--", lw=0.5)
ax.axvline(0, linestyle="--", lw=0.5)

ax.grid(False)
sb.despine()

ax.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.savefig("../../05_plots/16_CD4analysis/23_CD4_CD8_overlapping_DEGs_CSFbias.pdf", format="pdf", dpi=400)
plt.show()

In [ ]:
df[df["names"].isin(["PRF1"])]

# B-LCL reactive clones

In [ ]:
## Load B-LCL reactive TCRb chains
clone_raw = pd.read_csv("../../06_csv_outputs/Clones/2026_4_BLCL_reactive_d19.csv")
clone_raw

In [ ]:
## Make a list of B_LCL reactive TCRb chains
# clone_list_BLCL1 = clone_raw["aaSeqCDR3"].drop_duplicates().tolist()
# clone_list_BLCL2 = clone_raw[clone_raw["FC0_19"] > 10]["aaSeqCDR3"].drop_duplicates().tolist()
# clone_list_BLCL3 = clone_raw[clone_raw["FC0_19"] > 100]["aaSeqCDR3"].drop_duplicates().tolist()

## Annotate sdata["gex"].obs
# sdata["gex"].obs["LCL_reactivity1"] = pd.Categorical(np.where(sdata["gex"].obs["IR_VDJ_1_junction_aa"].isin(clone_list_BLCL1), "B-LCL reactive", "Non-reactive"))
# sdata["gex"].obs["LCL_reactivity2"] = pd.Categorical(np.where(sdata["gex"].obs["IR_VDJ_1_junction_aa"].isin(clone_list_BLCL2), "B-LCL reactive", "Non-reactive"))
# sdata["gex"].obs["LCL_reactivity3"] = pd.Categorical(np.where(sdata["gex"].obs["IR_VDJ_1_junction_aa"].isin(clone_list_BLCL3), "B-LCL reactive", "Non-reactive"))

## Make a list of B_LCL reactive TCRb chains
clone_list_BLCL1 = clone_raw["Clone_handle"].drop_duplicates().tolist()
clone_list_BLCL2 = clone_raw[clone_raw["FC0_19"] > 10]["Clone_handle"].drop_duplicates().tolist()
clone_list_BLCL3 = clone_raw[clone_raw["FC0_19"] > 100]["Clone_handle"].drop_duplicates().tolist()

#define a clone_handle_beta (donor + TCRb_1)
sdata["gex"].obs["clone_handle_beta"] = sdata["gex"].obs["donor"].astype(str) + "_" + sdata["gex"].obs["IR_VDJ_1_junction_aa"].astype(str)

## Annotate sdata["gex"].obs
sdata["gex"].obs["LCL_reactivity1"] = pd.Categorical(np.where(sdata["gex"].obs["clone_handle_beta"].isin(clone_list_BLCL1), "B-LCL reactive", "Non-reactive"))
sdata["gex"].obs["LCL_reactivity2"] = pd.Categorical(np.where(sdata["gex"].obs["clone_handle_beta"].isin(clone_list_BLCL2), "B-LCL reactive", "Non-reactive"))
sdata["gex"].obs["LCL_reactivity3"] = pd.Categorical(np.where(sdata["gex"].obs["clone_handle_beta"].isin(clone_list_BLCL3), "B-LCL reactive", "Non-reactive"))
sdata.update()

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig_umap = sc.pl.umap(sdata["gex"], color=["LCL_reactivity1", "LCL_reactivity2", "LCL_reactivity3"], groups=["B-LCL reactive"], size = 10, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, cmap="RdBu_r", return_fig=True)
fig_umap.savefig( "../../05_plots/16_CD4analysis/26_umap_B_LCL_reactive.pdf", bbox_inches="tight", format="pdf", dpi=400)
plt.show()

In [ ]:
sc.settings.set_figure_params(figsize=(4.3, 4))
fig_umap = sc.pl.umap(sdata["gex"], color=["donor"], groups=["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"], size = 10, legend_fontweight= 'bold', frameon= False ,legend_fontoutline=2, show=False, cmap="RdBu_r", return_fig=True)
fig_umap.savefig( "../../05_plots/15_CD4annotation/18_umap_B_LCL_reactive_donor.pdf", bbox_inches="tight", format="pdf", dpi=400)
plt.show()

In [ ]:
## groups
df = sdata["gex"].obs[(sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])) & (sdata["gex"].obs["Compartment"] == "PB")][["LCL_reactivity1", "L3_ann", "leiden_1.7"]].value_counts().reset_index()
df = df[~df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"])]
df["L3_ann"] = df["L3_ann"].cat.remove_unused_categories()
df["leiden_1.7"] = df["leiden_1.7"].cat.remove_unused_categories()

## Percentages
df["freq"] = df["count"] / df.groupby("LCL_reactivity1")["count"].transform("sum")

# Set desired order
order = ["Non-reactive", "B-LCL reactive"]

df["LCL_reactivity1"] = pd.Categorical(
    df["LCL_reactivity1"],
    categories=order,
    ordered=True
)

# Sort so pivot respects order
df = df.sort_values("LCL_reactivity1")

# Pivot to remove the background percentage
df_pivot = df.pivot(index="leiden_1.7", columns="LCL_reactivity1", values="freq")
df_pivot["delta"] = df_pivot["B-LCL reactive"] - df_pivot["Non-reactive"]
df_pivot = df_pivot.sort_values("delta").reset_index()

# --- plot ---
plt.figure(figsize=(5, 3.5))

sb.barplot(
    data=df_pivot,
    x="leiden_1.7",
    y="delta",
    color="lightgrey"   # base color (we'll override below if needed)
)
                                                                
# Add horizontal reference line at 0
plt.axhline(0, color="black", linestyle="--", linewidth=1)

plt.xticks()
plt.ylabel("Δ Fraction\n(B-LCL reactive - Non-reactive)")
plt.xlabel("")
plt.ylim(-0.2, 0.2)

sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig( "../../05_plots/16_CD4analysis/28_NormalizedColumnPlot.pdf", bbox_inches="tight", format="pdf", dpi=400)
plt.show()

# # Pivot for stacked barplot
# df_pivot = df.pivot(index="LCL_reactivity1", columns="L3_ann", values="freq")

# #Pass colors to the df
# colors = [color_dict2[col] for col in df_pivot.columns]

# df_pivot.plot(kind="bar", stacked=True, figsize=(8.25,6), color=colors)

# plt.ylabel("Frequency")
# plt.xlabel("")
# plt.title("Compartment = PB")
# plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="L3_ann")
# sb.despine()
# plt.grid(False)
# plt.tight_layout()
# plt.savefig( "../../05_plots/16_CD4analysis/27_StackedBarPB.pdf", bbox_inches="tight", format="pdf", dpi=400)
# plt.show()

In [ ]:
## groups
df = sdata["gex"].obs[(sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])) & (sdata["gex"].obs["Compartment"] == "CSF")][["LCL_reactivity1", "L3_ann", "leiden_1.7"]].value_counts().reset_index()
df = df[df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"])]
df["L3_ann"] = df["L3_ann"].cat.remove_unused_categories()
df["leiden_1.7"] = df["leiden_1.7"].cat.remove_unused_categories()


## Percentages
df["freq"] = df["count"] / df.groupby("LCL_reactivity1")["count"].transform("sum")

# Set desired order
order = ["Non-reactive", "B-LCL reactive"]

df["LCL_reactivity1"] = pd.Categorical(
    df["LCL_reactivity1"],
    categories=order,
    ordered=True
)

# Sort so pivot respects order
df = df.sort_values("LCL_reactivity1")
# Pivot to remove the background percentage
df_pivot = df.pivot(index="leiden_1.7", columns="LCL_reactivity1", values="freq")
df_pivot["delta"] = df_pivot["B-LCL reactive"] - df_pivot["Non-reactive"]
df_pivot = df_pivot.sort_values("delta").reset_index()

# --- plot ---
plt.figure(figsize=(3.65, 3.5))

sb.barplot(
    data=df_pivot,
    x="leiden_1.7",
    y="delta",
    color="lightgrey"   # base color (we'll override below if needed)
)
                                                                
# Add horizontal reference line at 0
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.ylim(-0.2, 0.2)
plt.xticks()
plt.ylabel("Δ Fraction\n(B-LCL reactive - Non-reactive)")
plt.xlabel("")
# plt.title("Delta per leiden clusters")

sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig( "../../05_plots/16_CD4analysis/28_NormalizedColumnPlot2.pdf", bbox_inches="tight", format="pdf", dpi=400)
plt.show()


# # Pivot for stacked barplot
# df_pivot = df.pivot(index="LCL_reactivity1", columns="L3_ann", values="freq")

# #Pass colors to the df
# colors = [color_dict2[col] for col in df_pivot.columns]

# df_pivot.plot(kind="bar", stacked=True, figsize=(8,6), color=colors)


# plt.ylabel("Frequency")
# plt.xlabel("")
# plt.title("Compartment = CSF")
# plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="L3_ann")
# sb.despine()
# plt.grid(False)
# plt.tight_layout()
# plt.savefig( "../../05_plots/16_CD4analysis/27_StackedBarCSF.pdf", bbox_inches="tight", format="pdf", dpi=400)
# plt.show()

# CSF-biased score for B-LCL reactive cells

In [ ]:
## PER CELL
df = sdata["gex"].obs[(sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])) & (sdata["gex"].obs["Compartment"] == "PB")][["catpat", "clone_handle", "LCL_reactivity1", "L3_ann", "leiden_1.7", "CSF permissive score"]].value_counts().reset_index()
df = df[~df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"])]
df["L3_ann"] = df["L3_ann"].cat.remove_unused_categories()
df["leiden_1.7"] = df["leiden_1.7"].cat.remove_unused_categories()

# Define fill colors
fill_palette = {
    "Control": "#FFFFFF",   # white
    "RRMS": "#000000"       # black
}

plt.figure(figsize=(6, 6))

# --- Points ---
sb.stripplot(
    data=df,
    x="LCL_reactivity1",
    y="CSF permissive score",
    hue="catpat",
    dodge=True,
    jitter=0.2,
    alpha=1,
    size=4,
    edgecolor="black",
    palette=fill_palette,
    linewidth=0.3,
    zorder=2
)

# --- Boxplot ---
sb.boxplot(
    data=df,
    x="LCL_reactivity1",
    y="CSF permissive score",
    hue="catpat",
    dodge=True,
    showcaps=True,
    showfliers=False,   
    palette=palette,
    boxprops={"alpha": 0.5},   
    whiskerprops={"linewidth": 1.2, "color" : "black"},
    medianprops={"color": "black"},
    zorder=1
)

# --- Fix duplicated legend ---
handles, labels = plt.gca().get_legend_handles_labels()
n = df["catpat"].nunique()  # <-- correct variable
plt.legend(
    handles[:n],
    labels[:n],
    title="catpat",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.ylabel("CSF permissive score")
plt.xlabel("")
plt.title("CSF permissive score per cell")

sb.despine()
plt.grid(False)
plt.tight_layout()
plt.show()

print(df[["catpat", "LCL_reactivity1"]].value_counts())

In [ ]:
# Mean per clone
df = sdata["gex"].obs[(
    (sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])) & 
    (sdata["gex"].obs["Compartment"] == "PB") #&
    # (sdata["gex"].obs["type_x_clone_count"] > 0)
    )][["catpat", "clone_handle", "LCL_reactivity1", "L3_ann", "leiden_1.7", "CSF permissive score"]].value_counts().reset_index()
df = df[~df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"])]
df["L3_ann"] = df["L3_ann"].cat.remove_unused_categories()
df["leiden_1.7"] = df["leiden_1.7"].cat.remove_unused_categories()

df = df.groupby(["LCL_reactivity1", "catpat", "clone_handle"], observed=True)["CSF permissive score"].mean().reset_index()

plt.figure(figsize=(6, 6))
# --- Points ---
sb.stripplot(
    data=df,
    x="LCL_reactivity1",
    y="CSF permissive score",
    hue="catpat",
    dodge=True,
    jitter=0.2,
    alpha=1,
    size=7,
    edgecolor="black",
    palette=fill_palette,
    linewidth=0.3,
    zorder=2
)

# --- Boxplot ---
sb.boxplot(
    data=df,
    x="LCL_reactivity1",
    y="CSF permissive score",
    hue="catpat",
    dodge=True,
    showcaps=True,
    showfliers=False,   
    palette=palette,
    boxprops={"alpha": 0.5},   
    whiskerprops={"linewidth": 1.2, "color" : "black"},
    medianprops={"color": "black"},
    zorder=1
)

# --- Fix duplicated legend ---
handles, labels = plt.gca().get_legend_handles_labels()
n = df["catpat"].nunique()  # <-- correct variable
plt.legend(
    handles[:n],
    labels[:n],
    title="catpat",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.ylabel("CSF-biased score")
plt.xlabel("")
plt.title("CSF permissive score per clone")

sb.despine()
plt.grid(False)
plt.tight_layout()
plt.savefig("../../05_plots/16_CD4analysis/29_CD4_BLCL_CSFbiased_per_clone.pdf", format="pdf", dpi=400)
plt.show()

print(df[["catpat", "LCL_reactivity1"]].value_counts())


In [ ]:
## PER DONOR
df = sdata["gex"].obs[(sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])) & (sdata["gex"].obs["Compartment"] == "PB")][["catpat", "clone_handle", "LCL_reactivity1", "L3_ann", "leiden_1.7", "donor", "CSF permissive score"]].value_counts().reset_index()
df = df[~df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"])]
df["L3_ann"] = df["L3_ann"].cat.remove_unused_categories()
df["leiden_1.7"] = df["leiden_1.7"].cat.remove_unused_categories()

df = df.groupby(["LCL_reactivity1", "catpat", "donor"], observed=True)["CSF permissive score"].mean().reset_index()

plt.figure(figsize=(6, 6))

plt.figure(figsize=(6, 6))
# --- Points ---
sb.stripplot(
    data=df,
    x="LCL_reactivity1",
    y="CSF permissive score",
    hue="catpat",
    dodge=True,
    jitter=0.2,
    alpha=1,
    size=7,
    edgecolor="black",
    palette=fill_palette,
    linewidth=0.3,
    zorder=2
)

# --- Boxplot ---
sb.boxplot(
    data=df,
    x="LCL_reactivity1",
    y="CSF permissive score",
    hue="catpat",
    dodge=True,
    showcaps=True,
    showfliers=False,   
    palette=palette,
    boxprops={"alpha": 0.5},   
    whiskerprops={"linewidth": 1.2, "color" : "black"},
    medianprops={"color": "black"},
    zorder=1
)

# --- Fix duplicated legend ---
handles, labels = plt.gca().get_legend_handles_labels()
n = df["catpat"].nunique()  # <-- correct variable
plt.legend(
    handles[:n],
    labels[:n],
    title="catpat",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.ylabel("CSF permissive score")
plt.xlabel("")
plt.title("CSF permissive score per cell")

sb.despine()
plt.grid(False)
plt.tight_layout()
plt.show()

print(df[["catpat", "LCL_reactivity1"]].value_counts())


# Graveyard

In [ ]:
## groups
df = sdata["gex"].obs[(sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])) & (sdata["gex"].obs["Compartment"] == "CSF")][["LCL_reactivity1", "L3_ann", "leiden_1.7"]].value_counts().reset_index()
df = df[df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"])]
df["L3_ann"] = df["L3_ann"].cat.remove_unused_categories()

## Percentages
df["freq"] = df["count"] / df.groupby("LCL_reactivity2")["count"].transform("sum")

# Set desired order
order = ["Non-reactive", "B-LCL reactive"]

df["LCL_reactivity2"] = pd.Categorical(
    df["LCL_reactivity2"],
    categories=order,
    ordered=True
)

# Sort so pivot respects order
df = df.sort_values("LCL_reactivity2")

# Pivot for stacked barplot
df_pivot = df.pivot(index="LCL_reactivity2", columns="L3_ann", values="freq")

df_pivot.plot(kind="bar", stacked=True, figsize=(8,6))

plt.ylabel("Frequency")
plt.xlabel("")
plt.title("Compartment = CSF")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="L3_ann")
sb.despine()
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
## groups
df = sdata["gex"].obs[(sdata["gex"].obs["donor"].isin(["Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"])) & (sdata["gex"].obs["Compartment"] == "CSF")][["LCL_reactivity3", "L3_ann", "leiden_1.7"]].value_counts().reset_index()
df = df[df["leiden_1.7"].isin(["6", "7", "8", "9", "10", "15", "16", "19"])]
df["L3_ann"] = df["L3_ann"].cat.remove_unused_categories()

## Percentages
df["freq"] = df["count"] / df.groupby("LCL_reactivity3")["count"].transform("sum")

# Set desired order
order = ["Non-reactive", "B-LCL reactive"]

df["LCL_reactivity3"] = pd.Categorical(
    df["LCL_reactivity3"],
    categories=order,
    ordered=True
)

# Sort so pivot respects order
df = df.sort_values("LCL_reactivity3")

# Pivot for stacked barplot
df_pivot = df.pivot(index="LCL_reactivity3", columns="L3_ann", values="freq")

df_pivot.plot(kind="bar", stacked=True, figsize=(8,6))

plt.ylabel("Frequency")
plt.xlabel("")
plt.title("Compartment = CSF")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="L3_ann")
sb.despine()
plt.grid(False)
plt.tight_layout()
plt.show()

# clonal overlap d19 and sc-seq clusters

In [ ]:
## Reproduce Kavaka et al analysis with clonal overlap among each cluster
#### Of all the CSF clonotypes - which cluster contains the most overlap?

obs_df = sdata["gex"].obs[["donor", "Compartment", "catpat", "leiden_1.7", "IR_VDJ_1_junction_aa"]].copy()
obs_df["clone_handle"] = obs_df["donor"].astype(str) + "_" + obs_df["IR_VDJ_1_junction_aa"].astype(str)
obs_df = obs_df[~obs_df["clone_handle"].isna()]

## TCSF cells
obs_csf = obs_df[obs_df["Compartment"]=="CSF"]
list_clones = clone_raw["Clone_handle"].drop_duplicates().tolist()

# Ensure CSF clones is a set
csf_clones = set(list_clones)

# Define the clusters to iterate over and the cluster column. 
# These clusters has to be PB clusters
clusters =  ["0", "5", "1", "3", "11", "2", "4", "12", "18", "14", "13", "17", "20", "6", "7", "8", "9", "10", "15", "16", "19"]
cluster_col = "leiden_1.7"

# Define an empty dataframe
results = []

for cl in clusters:
    # Subset for given cluster in PB
    sub = obs_df[
        (obs_df[cluster_col] == cl) 
        # &         (obs_df["Compartment"] == "PB")
    ]

    cluster_clones = set(sub["clone_handle"].drop_duplicates())

    # Overlap and union
    overlap = csf_clones & cluster_clones
    union = csf_clones | cluster_clones

    pct_csf_in_cluster = (
        len(overlap) / len(csf_clones) * 100
        if len(csf_clones) > 0 else np.nan
    )

    jaccard = (
        len(overlap) / len(union) * 100
        if len(union) > 0 else np.nan
    )

    results.append({
        cluster_col: cl,
        "n_csf_clones": len(csf_clones),
        "n_cluster_clones": len(cluster_clones),
        "n_overlap": len(overlap),
        "pct_csf_in_cluster": pct_csf_in_cluster,
        "jaccard_index": jaccard
    })

df_results = pd.DataFrame(results)
df_results



In [ ]:
## Make a column plot that describes clonal sharing amongst CD8 T cells
# Ensure correct ordering
# order = ["TN", "TSCM", "TCM", "TEM", "TE", "TAct"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot 1: % CSF in cluster ---
sb.barplot(
    data=df_results,
    x="leiden_1.7",
    y="pct_csf_in_cluster",
    # order=order,
    ax=axes[0]
)

axes[0].set_title("% CSF clones in cluster")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Percentage")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(False)

# --- Plot 2: Jaccard index ---
sb.barplot(
    data=df_results,
    x="leiden_1.7",
    y="jaccard_index",
    # order=order,
    ax=axes[1]
)

axes[1].set_title("Jaccard index")
axes[1].set_xlabel("PB cluster")
axes[1].set_ylabel("Percentage")
axes[1].tick_params(axis="x", rotation=0)
axes[1].grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/16_CD4analysis/25_CD4_BLCLreac_Jaccard_overlap_leiden_1.7.pdf")
plt.show()

# Try pseudobulk analysis using Decoupler
https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_psbk.html

The pseudo-bulk approach involves the following steps:

    Subsetting the cell type of interest

    Extracting their raw integer counts

    Summing their counts per gene into a single profile if they pass quality control

Then, DEA can be performed if there are at least two biological replicates per condition (more replicates are recommended).

Pseudobulking can easily be performed using the function decoupler.pp.pseudobulk(). In this example, the counts are just summed, though other modes such as the mean or any custom aggregation function are available. For more information, refer to the mode argument.

In [ ]:
## ---- Define the data ---- 
sdata_bulk = sdata.copy()

mu.pp.filter_obs(sdata_bulk["gex"], "Compartment", lambda x: (x == "PB")) 
mu.pp.filter_obs(sdata_bulk["gex"], "top_vs_bot_clones", lambda x: (x != "unselected")) 
sdata_bulk["gex"].obs["top_vs_bot_clones"] = sdata_bulk["gex"].obs["top_vs_bot_clones"].cat.remove_unused_categories()

# Create a mask for genes to exclude
genes = sdata_bulk["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_bulk = sdata_bulk["gex"][:, genes_to_keep].copy()

## Bring the raw counts into .X
sdata_bulk.X = sdata_bulk.layers["counts"].copy()

In [ ]:
## ---- Use decoupler to pseudobulk cells ----
pdata = dc.pp.pseudobulk(
    adata=sdata_bulk,
    sample_col="donor",
    groups_col="top_vs_bot_clones",
    mode="sum",
)

pdata

In [ ]:
pdata.obs

In [ ]:
dc.pp.filter_samples(pdata, min_cells=1, min_counts=1000)

In [ ]:
pdata.obs

In [ ]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=pdata,
    design_factors=["top_vs_bot_clones"],
    refit_cooks=True,
    inference=inference,
)

# Compute LFCs
dds.deseq2()

# Extract contrast between conditions
stat_res = DeseqStats(dds, contrast=["top_vs_bot_clones", "PB-biased", "CSF-biased"], inference=inference)

# Compute Wald test
stat_res.summary()

In [ ]:
# Extract results
results_df = stat_res.results_df
results_df.sort_values("log2FoldChange", ascending=False).head(20)

In [ ]:
dc.pl.volcano(results_df, x="log2FoldChange", y="pvalue", top=10, figsize =(6,6))
plt.show()

In [ ]:
data = results_df[["stat"]].T.rename(index={"stat": "Biased.vs.Not"})
data

# Try filtering using a custom script

### CSF biased vs PB biased

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Differential gene expression
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
sdata_pb = sdata.copy()

mu.pp.filter_obs(sdata_pb["gex"], "Compartment", lambda x: (x == "PB")) 
# --- I considered removing the naive clusters first to make filtering amongst memory cells more relevant later ---
mu.pp.filter_obs(sdata_pb["gex"], "L3_ann_overview", lambda x: (~x.isin(["TN"])))

# -- I considered removing the unselected, but the sc.tl.filter_rank_genes_groups() function also removes genes with high expression among unselected, which is desirable for finding genes specific to the 2 selected groups --- 
# mu.pp.filter_obs(sdata_pb["gex"], "top_vs_bot_clones", lambda x: (x != "unselected")) 
# sdata_pb["gex"].obs["top_vs_bot_clones"] = sdata_pb["gex"].obs["top_vs_bot_clones"].cat.remove_unused_categories()


## Remove TCR, Ribosomal genes before calculating DEGs
### Apply filtering to the DEGs

In [ ]:
# Remove TCR-related and ribosomal genes
# TCR genes often start with 'TRAV', 'TRBV', 'TRGV', etc.
# Ribosomal genes often start with 'RPS' or 'RPL'

sdata_pb_filtered = sdata_pb.copy()

# IMPORTANT TO SUBSET ONLY ON CELLS OF INTEREST- because the filter_rank_genes behaves odd if there are cells outside the groups (target and reference) that express a gene of interest. CCR7 kept getting filtered out because naive cells/other cells were part of the analysis
# mu.pp.filter_obs(sdata_pb_filtered["gex"], "top_vs_bot_clones", lambda x: (x != "unselected"))   #Going back to using memory cells as background for filtering

# Create a mask for genes to exclude
genes = sdata_pb["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_pb_filtered = sdata_pb_filtered["gex"][:, genes_to_keep]

# Run rank_genes_groups with filtered genes
sc.tl.rank_genes_groups(
    sdata_pb_filtered,
    groupby="top_vs_bot_clones",
    groups=["CSF-biased"],
    reference="PB-biased",
    method="wilcoxon",
    key_added="CSF_biased_gene_set"
)

sc.tl.rank_genes_groups(
    sdata_pb_filtered,
    groupby="top_vs_bot_clones",
    groups=["PB-biased"],
    reference="CSF-biased",
    method="wilcoxon",
    key_added="PB_biased_gene_set"
)

sc.pl.rank_genes_groups_dotplot(
      sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=80, key="CSF_biased_gene_set")
sc.pl.rank_genes_groups_dotplot(
      sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=80, key="PB_biased_gene_set")

# %%%% 
# Export all differentially expressed genes
# %%%%

Export_DEG = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="CSF_biased_gene_set")
# Export_DEG.to_csv("..\\..\\06_csv_outputs\\DEG\\2025_5_CD8_CSF_DEG_without_ribosomal_or_mitoch_genes.csv")

In [ ]:
# %%% 
# Vulcano plot function
# %%%

# # From: https://www.sc-best-practices.org/conditions/differential_gene_expression.html
LOG_FOLD_CHANGE = 1
PVALUE = -np.log10(0.01)  #Higher than p_adj 0.05. 
Add_labels = []
Add_labels = ['GZMK', 'CCR7', 'CD27','GPR183', 'CXCR4', "CXCR3", "CCR5", "MS4A1", "AC004448.2", 
               # 'CLDND1', 'COTL1',  'ITM2C',  'SARAF',  'NELL2',
              'GZMB', 'ZNF683', 'GZMH', 'KLRD1',  'CX3CR1', 'FNDC3B',  'KLRC2', "TYROBP", 
              # 'LINC02384', 'LINC00278',# 
             ]

VOLCANO_PALETTE = {
    "neg": "#4B8AC1",   # e.g. CSF-biased (negative logFC)
    "pos": "#C06A6A",   # e.g. PB-biased (positive logFC)
    "ns":  "lightgrey"
}
def volcano_plot(adata, group_key, title=None, palette=VOLCANO_PALETTE):
    result = sc.get.rank_genes_groups_df(
        adata, group=None, key=group_key
    ).copy()

    result["-log_padj"] = -np.log10(result["pvals_adj"].astype(float))

    #Remove dots with a very low log2fc
    result=result[(result["logfoldchanges"] > 1) | (result["logfoldchanges"] < -1)]

    #Drop stuff that is na
    result = result.dropna(subset="names")

    # Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_padj"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_padj"] <= PVALUE)
    ]

    # Split significant genes by direction
    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.grid(True, zorder=0)

    # Non-significant genes
    sb.regplot(
        x=nonsig["logfoldchanges"],
        y=nonsig["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "alpha": 0.5, "zorder": 2},
        ax=ax,
        color=palette["ns"]
    )

    # Positively enriched
    sb.regplot(
        x=sig_pos["logfoldchanges"],
        y=sig_pos["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["pos"],
        label="Positive enrichment"
    )

    # Negatively enriched
    sb.regplot(
        x=sig_neg["logfoldchanges"],
        y=sig_neg["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["neg"],
        label="Negative enrichment"
    )

    # Rasterize only the dots
    for coll in ax.collections:
        coll.set_rasterized(True)

    # Gene labels
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        ax.text(
            row["logfoldchanges"],
            row["-log_padj"],
            row["names"],
            fontsize=8,
            ha="right",
            va="bottom",
            zorder=4
        )

    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log padj")

    if title is None:
        title = group_key.replace("_", " ")
    ax.set_title(title)

    ax.legend(frameon=False)
    plt.tight_layout()

    # plt.savefig(
    #     "../../05_plots/13_final_plots/17_CD8_vulcanoPlot_CSFvPB_bias.pdf",
    #     bbox_inches="tight",
    #     format="pdf",
    #     dpi=400
    # )
    plt.show()

volcano_plot(sdata_pb_filtered, group_key="PB_biased_gene_set", title="<-- CSF-biased & PB-biased -->") 

# Export raw CSF-biased DEGs

In [ ]:
## Get two dataframes of positive lfc values associated with CSF-biased and PB-biased T cells. Concatenate them and plot
datCSF_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="CSF_biased_gene_set").dropna(subset="names")
datPB_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="PB_biased_gene_set").dropna(subset="names")

#EXPORT unfiltered gene scores
# datCSF_raw.to_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_raw_DEGs.csv")
# datPB_raw.to_csv("../../06_csv_outputs/DEG/CD8_PB_biased_raw_DEGs.csv")

# Filter DEGs
We filter the list of differentially expressed genes because the wilcox tests are prone to false positives 

### List of genes that pass filtering thresholds

**Note:** Setting "pts = True" enables calculation of % of cells in the given groups that express a given gene

In [ ]:
## To get the fractions for each gene in CSF-biased and unselected I also run a comparison against all unselected cells
sc.tl.rank_genes_groups(sdata_pb_filtered, groupby="top_vs_bot_clones", groups=["CSF-biased"], reference="unselected", n_genes=50, key_added="CSF_fra", pts = True, method="wilcoxon")

In [ ]:
# Get a gene list ready for filtering the list of differentially expressed genes where:
## (A)  The fraction of cells expressing CSF is high in CSF-biased cells (higher than 10%)
## (B)  The fraction of cells outside CSF (i.e. unselected) is less than PB  (!! this is different from scanpy's filtering function that compares X group vs "rest" !!) 
res = sdata_pb_filtered.uns["CSF_fra"]

df = pd.DataFrame({
    "CSF_fra" : res["pts"]["CSF-biased"],
    "uslct_fra" : res["pts"]["unselected"],
})

## (A)
list_10 = df[df["CSF_fra"] > 0.1].index.tolist()

## (B) 
list_unslct = df[df["CSF_fra"] > df["uslct_fra"]].index.tolist()

## Generate a set
list_CSFbiased = set(list_10) & set(list_unslct)

In [ ]:
## To get the fractions for each gene in CSF-biased and unselected I also run a comparison against all unselected cells
sc.tl.rank_genes_groups(sdata_pb_filtered, groupby="top_vs_bot_clones", groups=["PB-biased"], reference="unselected", n_genes=50, key_added="PB_fra", pts = True, method="wilcoxon")

In [ ]:
# Get a gene list ready for filtering the list of differentially expressed genes where:
## (A)  The fraction of cells expressing CSF is high in CSF-biased cells (higher than 10%)
## (B)  The fraction of cells outside CSF (i.e. unselected) is less than PB  (!! this is different from scanpy's filtering function that compares X group vs "rest" !!) 
res =sdata_pb_filtered.uns["PB_fra"]

df = pd.DataFrame({
    "PB_fra" : res["pts"]["PB-biased"],
    "uslct_fra" : res["pts"]["unselected"],
})

## (A)
list_10 = df[df["PB_fra"] > 0.1].index.tolist()

## (B) 
list_unslct = df[df["PB_fra"] > df["uslct_fra"]].index.tolist()

## Generate a set
list_PBbiased = set(list_10) & set(list_unslct)

In [ ]:
df[df.index == "KIR2DL1"]

In [ ]:
df[df.index == "AC004448.2"]

# Filter using custom thresholds (A) and (B) from above

In [ ]:
## Get two dataframes of positive lfc values associated with CSF-biased and PB-biased T cells. Concatenate them and plot
datCSF_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="CSF_biased_gene_set").dropna(subset="names")
datPB_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="PB_biased_gene_set").dropna(subset="names")

## Filter using the names in list_CSFbiased
datCSF_filtered = datCSF_raw[datCSF_raw["names"].isin(list_CSFbiased)]
datPB_filtered = datPB_raw[datPB_raw["names"].isin(list_PBbiased)]

# datPB_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="PB_biased_gene_set").dropna(subset="names")
# #EXPORT unfiltered gene scores
# datCSF_raw.to_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_raw_DEGs.csv")
# datPB_raw.to_csv("../../06_csv_outputs/DEG/CD8_PB_biased_raw_DEGs.csv")

### Filtered vulcano plot

In [ ]:
## get data frames
datCSF_filtered
datPB_filtered

#Report 
print("Number of CSF-biased DEGs: ", len(datCSF_filtered[(datCSF_filtered["logfoldchanges"]> 1) & (datCSF_filtered["pvals"]<0.05)])) 
print("Number of PB-biased DEGs: ", len(datPB_filtered[(datPB_filtered["logfoldchanges"]> 1) & (datPB_filtered["pvals"]<0.05)])) 

## Remove NaN genes 
# datCSF = datCSF.dropna(subset="names")
datCSF_filtered["cat_score"] = "CSF_score"
# datPB = datPB.dropna(subset="names")
datPB_filtered["cat_score"] = "PB_score"

#Concatenate
datDEG = pd.concat([datPB_filtered, datCSF_filtered])

#Remove negative l2fc values (these are present twice in the final dataframe)
datDEG = datDEG[datDEG["logfoldchanges"] > 0]

#Set all logfold change values from CSF-biased T cells to minus. 
datDEG["logfoldchanges"] = np.where(datDEG["cat_score"] == "CSF_score", -1*datDEG["logfoldchanges"] , datDEG["logfoldchanges"])

# How many unique genes?
print("I have this many unique genes: ", len(datDEG["names"].drop_duplicates()))
print("My dataframe is this long: ", len(datDEG))

In [ ]:
LOG_FOLD_CHANGE = 1
PVALUE = -np.log10(0.05) 
Add_labels = []
Add_labels = ['GZMK', 'CCR7', 'CD27','GPR183', 'CXCR4', "CXCR3", "CCR5", "MS4A1", 'CLDND1',  'ITM2C',  'SARAF',  'NELL2', "DKK3", "PASK", "COTL1", "CD28", "PDCD1", 
 'PPP1R14B', 'CAMK4', 'MCUB', 'RGCC', 'PDE4B', 'PHACTR2', 'DUSP2', 'CMC1', 'TRAT1', 'SH2D1A', 'DTHD1', 'OCIAD2', 'ST8SIA1', 'LEPROTL1', "IFNG-AS1",  "TNFSF8", "DUSP4", "ADARB2", "CRTAM", "ICOS", 
              "LINC02384", "LINC00278", 
              'GZMB', 'ZNF683', 'GZMH', 'KLRD1',  'CX3CR1', 'FNDC3B',  'KLRC2', "ASCL2",  "GPR141", "ABCB1", "PRF1", "KIR3DL1", "KIR3DL2", "TYROBP", "GNLY", "NCR3", "KLRF1", "AC004448.2", 
              "HOPX", "ITGB1", "S100A4", "EFHD2", "FCRL6", "FCGR3A", "NKG7", "CTSW", "MATK", "S1PR5", 
             ]

VOLCANO_PALETTE = {
    "neg": "#8A7DBF",   # e.g. CSF-biased (negative logFC)
    "pos": "#C06A6A",   # e.g. PB-biased (positive logFC)
    "ns":  "lightgrey"
}

def volcano_plot_simple(dat_df, title=None, palette=VOLCANO_PALETTE):
    result=dat_df.copy()
    result["-log_p"] = -np.log10(result["pvals"].astype(float))

    #Remove dots with a very low log2fc
    result=result[(result["logfoldchanges"] > 0.5) | (result["logfoldchanges"] < -0.5)]

    # Selecting values to highlight - Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_p"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_p"] <= PVALUE)
    ]

    # Split significant genes by direction
    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    fig, ax = plt.subplots(figsize=(6, 6)) #6.6
    ax.grid(True, zorder=0)

    # Non-significant genes
    sb.regplot(
        x=nonsig["logfoldchanges"],
        y=nonsig["-log_p"],
        fit_reg=False,
        scatter_kws={"s": 60, "alpha": 0.5, "zorder": 2},
        ax=ax,
        color=palette["ns"]
    )

    # Positively enriched
    sb.regplot(
        x=sig_pos["logfoldchanges"],
        y=sig_pos["-log_p"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["pos"],
        label="Positive enrichment"
    )

    # Negatively enriched
    sb.regplot(
        x=sig_neg["logfoldchanges"],
        y=sig_neg["-log_p"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["neg"],
        label="Negative enrichment"
    )

    # # Rasterize only the dots
    # for coll in ax.collections:
    #     coll.set_rasterized(True)

    # Gene labels
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        ax.text(
            row["logfoldchanges"],
            row["-log_p"],
            row["names"],
            fontsize=8,
            ha="right",
            va="bottom",
            zorder=4
        )

    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log p")
    # ax.set_xlim(-6.25,6.25)

    if title is None:
        title = group_key.replace("_", " ")
    ax.set_title(title)

    ax.legend(frameon=False)
    sb.despine()
    ax.grid(False)
    plt.tight_layout()

    # plt.savefig(
    #     "../../05_plots/13_final_plots/17_CD8_vulcanoPlot_CSFvPB_bias_afterfiltering_simple.pdf",
    #     bbox_inches="tight",
    #     format="pdf",
    #     dpi=400
    # )
    plt.show()

In [ ]:
volcano_plot_simple(datDEG, title="<-- CSF-biased & PB-biased -->") 

In [ ]:
datDEG.sort_values("logfoldchanges", ascending=False)

## Former filtering approach

In [ ]:
#Old filtering method:
## --- The custom filtering approach is inspired by sc.tl.filter_rank_genes_groups()
#      This scanpy function kept removing genes with higher expression outside the target group "rest". What I want is to only remove genes that are higher expressed among "unselected". 

sc.tl.filter_rank_genes_groups(
    sdata_pb_filtered,
    min_in_group_fraction=0.1,  #fraction positive in population of interest (i.e if I am looking at genes related to CSF-migration, then it should be expressed in X fraction of the target group
    max_out_group_fraction=1,   #Max is 1 - no filtering will take place
    min_fold_change=0,
    compare_abs=False,       #Compare_abs=True retains negative log2fc genes.. We specifically want upregulated genes for the next analysis
    key="CSF_biased_gene_set",
    key_added="CSF_biased_gene_set (w. filtering)",
)

sc.tl.filter_rank_genes_groups(
   sdata_pb_filtered,
    min_in_group_fraction=0.1, #fraction positive in population of interest 
    max_out_group_fraction=1,   #fraction positive !! outside !! the population of interest be less than 0.9
    min_fold_change=0,      #To retain all information
    compare_abs=False,       #Compare_abs=True retains negative log2fc genes..  We specifically want upregulated genes for the next analysis.
    key="PB_biased_gene_set",
    key_added="PB_biased_gene_set (w. filtering)",
)

#There seems to be a BUG in sc.tl.filter_rank_genes_groups(min_fold_change = 0, compare_abs=False) still retains negative fold change values. 
## must be removed later

p1=sc.pl.rank_genes_groups_dotplot(
      sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=15, key="CSF_biased_gene_set (w. filtering)", color_map='RdYlBu_r', return_fig=True)
p1.savefig("../../05_plots/16_CD4analysis/18_CD4_expressiondots_CSF_score.pdf", bbox_inches='tight', format="pdf", dpi=400)
p2=sc.pl.rank_genes_groups_dotplot(
      sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=15, key="PB_biased_gene_set (w. filtering)", color_map='RdYlBu_r', return_fig=True)
p2.savefig("../../05_plots/16_CD4analysis/18_CD4_expressiondots_CSF_score.pdf", bbox_inches='tight', format="pdf", dpi=400)

plt.show

## Get two dataframes of positive lfc values associated with CSF-biased and PB-biased T cells. Concatenate them and plot
datCSF = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="CSF_biased_gene_set (w. filtering)").dropna(subset="names")
datPB = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="PB_biased_gene_set (w. filtering)").dropna(subset="names")

#Report 
print("Number of CSF-biased DEGs: ", len(datCSF[(datCSF["logfoldchanges"]> 1) & (datCSF["pvals_adj"]<0.01)])) 
print("Number of PB-biased DEGs: ", len(datPB[(datPB["logfoldchanges"]> 1) & (datPB["pvals_adj"]<0.01)])) 

## Remove NaN genes 
# datCSF = datCSF.dropna(subset="names")
datCSF["cat_score"] = "CSF_score"
# datPB = datPB.dropna(subset="names")
datPB["cat_score"] = "PB_score"

#Concatenate
datDEG = pd.concat([datPB, datCSF])

#Remove negative l2fc values (these are present twice in the final dataframe)
datDEG = datDEG[datDEG["logfoldchanges"] > 0]

#Set all logfold change values from CSF-biased T cells to minus. 
datDEG["logfoldchanges"] = np.where(datDEG["cat_score"] == "CSF_score", -1*datDEG["logfoldchanges"] , datDEG["logfoldchanges"])

# How many unique genes?
print("I have this many unique genes: ", len(datDEG["names"].drop_duplicates()))
print("My dataframe is this long: ", len(datDEG))


LOG_FOLD_CHANGE = 1
PVALUE = -np.log10(0.01) 
Add_labels = []
Add_labels = ['GZMK', 'CCR7', 'CD27','GPR183', 'CXCR4', "CXCR3", "CCR5", "MS4A1", "CD28", "PDCD1", 
              "LINC02384", "LINC00278", 'GZMB', 'ZNF683', 'GZMH', 'KLRD1',  'CX3CR1', "PRF1", "KIR3DL1", "KIR3DL2", "TYROBP", "GNLY", "NCR3", "KLRF1", "ITGB1", "S100A4", "NKG7", "CTSW", "MATK", "S1PR5", 
             ]

VOLCANO_PALETTE = {
    "neg": "#8A7DBF",   # e.g. CSF-biased (negative logFC)
    "pos": "#C06A6A",   # e.g. PB-biased (positive logFC)
    "ns":  "lightgrey"
}

def volcano_plot_simple(dat_df, title=None, palette=VOLCANO_PALETTE):
    result=dat_df.copy()
    result["-log_padj"] = -np.log10(result["pvals_adj"].astype(float))

    #Remove dots with a very low log2fc
    result=result[(result["logfoldchanges"] > 0.5) | (result["logfoldchanges"] < -0.5)]

    # Selecting values to highlight - Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_padj"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_padj"] <= PVALUE)
    ]

    # Split significant genes by direction
    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, figsize=(6, 6), gridspec_kw={'width_ratios': [4, 1]})  
    
    for ax in [ax1, ax2]:
        ax.grid(True, zorder=0)
        ax.grid(True, zorder=0)

    def plot_data(ax):
        # Non-significant
        sb.regplot(
            x=nonsig["logfoldchanges"],
            y=nonsig["-log_padj"],
            fit_reg=False,
            scatter_kws={"s": 60, "alpha": 0.5, "zorder": 2},
            ax=ax,
            color=palette["ns"]
        )
    
        # Pos
        sb.regplot(
            x=sig_pos["logfoldchanges"],
            y=sig_pos["-log_padj"],
            fit_reg=False,
            scatter_kws={"s": 30, "zorder": 3},
            ax=ax,
            color=palette["pos"]
        )
    
        # Neg
        sb.regplot(
            x=sig_neg["logfoldchanges"],
            y=sig_neg["-log_padj"],
            fit_reg=False,
            scatter_kws={"s": 30, "zorder": 3},
            ax=ax,
            color=palette["neg"]
        )
    
    plot_data(ax1)
    plot_data(ax2)

    ax1.set_xlim(-8, 8)
    ax2.set_xlim(27, 31.5)
    ax1.spines['right'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.yaxis.set_visible(False)
    
    d = .01
    kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
    ax1.plot((1-d, 1+d), (-d, +d), **kwargs)
    ax1.plot((1-d, 1+d), (1-d, 1+d), **kwargs)
    kwargs.update(transform=ax2.transAxes)
    
    ax2.plot((-d, +d), (-d, +d), **kwargs)
    ax2.plot((-d, +d), (1-d, 1+d), **kwargs)

    #Gene labels
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        x = row["logfoldchanges"]
        y = row["-log_padj"]
    
        if x <= 6:
            ax = ax1
        elif x >= 26:
            ax = ax2
        else:
            continue  # skip anything in the cut region
    
        ax.text(x, y, row["names"], fontsize=6,
                ha="right", va="bottom", zorder=4)

    ax1.set_xlabel("log2 FC")
    ax1.set_ylabel("-log padj")
    ax2.set_xlabel("log2 FC")
    ax1.set_title(title)
    
    ax.legend(frameon=False)
    
    plt.tight_layout()

    # plt.savefig(
    #     "../../05_plots/16_CD4analysis/19_CD4_vulcanoPlot_CSFvPB_bias_afterfiltering_simple.pdf",
    #     bbox_inches="tight",
    #     format="pdf",
    #     dpi=400
    # )
    plt.show()

volcano_plot_simple(datDEG, title="<-- CSF-biased & PB-biased -->") 

CSFbiased_genescore = datCSF[(datCSF["logfoldchanges"]> 1) & (datCSF["pvals_adj"]<0.05)]
PBbiased_genescore = datPB[(datPB["logfoldchanges"]> 1) & (datPB["pvals_adj"]<0.05)]

sc.tl.score_genes(sdata["gex"], 
                      CSFbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='CSF permissive score')

sc.tl.score_genes(sdata["gex"], 
                      PBbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='PB resident score')

p1 = sc.pl.umap(
    sdata["gex"], 
    color=["CSF permissive score", "PB resident score", "CCR7"], 
    size=10, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
plt.show()
# p1.savefig("../../05_plots/16_CD4analysis/20_umap_biased_scores.pdf", bbox_inches='tight', format="pdf", dpi = 400)

# Custom filtering of DEGs

### List of genes that pass filtering thresholds

**Note:** Setting "pts = True" enables calculation of % of cells in the given groups that express a given gene

In [ ]:
## To get the fractions for each gene in CSF-biased and unselected I also run a comparison against all unselected cells
sc.tl.rank_genes_groups(sdata_pb_filtered, groupby="top_vs_bot_clones", groups=["CSF-biased"], reference="unselected", n_genes=50, key_added="CSF_fra", pts = True, method="wilcoxon")
sc.tl.rank_genes_groups(sdata_pb_filtered, groupby="top_vs_bot_clones", groups=["PB-biased"], reference="unselected", n_genes=50, key_added="PB_fra", pts = True, method="wilcoxon")

In [ ]:
# Get a gene list ready for filtering the list of differentially expressed genes where:
## (A)  The fraction of cells expressing CSF is high in CSF-biased cells (higher than 0.05)
## (B)  The fraction of cells expressing a given gene is lower in unselected cells
## (C)  The fraction of cells expressing a given gene in PB-biased cells is high (higher than 0.05)

## Data from CSFbiased vs unselected
res = sdata_pb_filtered.uns["CSF_fra"]
df1 = pd.DataFrame({
    "CSF_fra" : res["pts"]["CSF-biased"],
    "uslct_fra" : res["pts"]["unselected"],
})

## Data from PBbiased vs unselected
res = sdata_pb_filtered.uns["PB_fra"]
df2 = pd.DataFrame({
    "PB_fra" : res["pts"]["PB-biased"],
    "uslct_fra" : res["pts"]["unselected"],
})

## (A)
list_A = df1[df1["CSF_fra"] > 0.05].index.tolist()

## (B)
list_B = df1[df1["uslct_fra"] < df1["CSF_fra"]].index.tolist()

## (C)
list_C = df2[df2["PB_fra"] > 0.05].index.tolist()

## (D)
list_D= df2[df2["uslct_fra"] < df2["PB_fra"]].index.tolist()


## Generate sets of indexes for filtering
list_CSFbiased = set(list_A) & set(list_B)
list_PBbiased = set(list_C) & set(list_D)


In [ ]:
df1[df1.index.isin(["CD27", "CCR7", "CCR5", "CXCR3", "CXCR4", "MS4A1", "GAPDH", "XIST", "KIR3DL1", "EEF1A1", "EEF1B2"])]

In [ ]:
df2[df2.index.isin(["CD27", "CCR7", "GAPDH", "XIST", "KIR3DL1", "AC004448.2"])]

# Filter using custom thresholds (A),(B), and (C) from above

In [ ]:
## Get two dataframes of positive lfc values associated with CSF-biased and PB-biased T cells. Concatenate them and plot
datCSF_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="CSF_biased_gene_set").dropna(subset="names")
datPB_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="PB_biased_gene_set").dropna(subset="names")

## Filter using the names in list_CSFbiased
datCSF_filtered = datCSF_raw[datCSF_raw["names"].isin(list_CSFbiased)]
datPB_filtered = datPB_raw[datPB_raw["names"].isin(list_PBbiased)]

## Filter using a log2fc and p.value threshold
print("Number of CSF-biased DEGs: ", len(datCSF_filtered[(datCSF_filtered["logfoldchanges"]> 1) & (datCSF_filtered["pvals_adj"]<0.05)])) 
print("Number of PB-biased DEGs: ", len(datPB_filtered[(datPB_filtered["logfoldchanges"]> 1) & (datPB_filtered["pvals_adj"]<0.05)])) 
datCSF_filtered = datCSF_filtered[(datCSF_filtered["logfoldchanges"]> 1) & (datCSF_filtered["pvals_adj"]<0.05)]
datPB_filtered = datPB_filtered[(datPB_filtered["logfoldchanges"]> 1) & (datPB_filtered["pvals_adj"]<0.05)]

# datPB_raw = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None,key ="PB_biased_gene_set").dropna(subset="names")
# #EXPORT unfiltered gene scores
# datCSF_raw.to_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_raw_DEGs.csv")
# datPB_raw.to_csv("../../06_csv_outputs/DEG/CD8_PB_biased_raw_DEGs.csv")

# Umap CSF/PB scores

In [ ]:
print(f"{len(CSFbiased_genescore)} genes make up my CSF permissive score")
print(f"{len(PBbiased_genescore)} genes make up my PB resident score")

In [ ]:
sc.pl.dotplot(sdata_pb_filtered, CSFbiased_genescore["names"], groupby='top_vs_bot_clones', standard_scale="var", dendrogram=True, cmap="RdBu_r")

In [ ]:
sc.pl.dotplot(sdata_pb_filtered, PBbiased_genescore["names"], groupby='top_vs_bot_clones', dendrogram=True,standard_scale="var", cmap="RdBu_r")

In [ ]:
sc.tl.score_genes(sdata["gex"], 
                      CSFbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='CSF permissive score')

sc.tl.score_genes(sdata["gex"], 
                      PBbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='PB resident score')


In [ ]:
p1 = sc.pl.umap(
    sdata["gex"], 
    color=["CSF permissive score", "PB resident score"], 
    size=10, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
plt.show()
p1.savefig("../../05_plots/16_CD4analysis/20_umap_biased_scores.pdf", bbox_inches='tight', format="pdf", dpi = 400)